# Glance: Visual Progression & Results

The following screenshots illustrate the effectiveness of this Hierarchical Expert-in-the-Loop pipeline.

### 1. Initial Clustering (Without Mask)
<img src="https://i.ibb.co/S44Yzx1w/image.png" width="800">

**Analysis Note**: Without an agricultural mask, the GMM distributions are heavily skewed by non-agricultural targets such as urban settlements, barren land, and water bodies. The crop boundaries are noisy, and distinct crop phenologies are overwhelmed by the macro-level landcover variance.

---

### 2. Primary Clustering (With Mask, K=20)
<img src="https://i.ibb.co/Df14QTGH/image.png" width="800">

**Analysis Note**: By constraining the analysis strictly to the expert-verified mask, the GMM forces its covariance matrices to model *only* vegetated cropland variations. However, at a high K value like K=20 globally, some clusters still represent "mixed" crop groups due to spectral overlap. A specific mixed cluster from this stage is selected as the `target_cluster` for hierarchical refinement.

---

### 3. Hierarchical Reclustering Results
<img src="https://i.ibb.co/Mxy8HzkG/image.png" width="800">
<br><br>
<img src="https://i.ibb.co/bj0YdwRy/Whats-App-Image-2026-07-24-at-10-25-50-PM.jpg" width="800">

**Analysis Note**: By freezing a specific mixed target cluster and recursively applying a fresh GMM exclusively on its pixels, we free the covariance matrices to model the subtle, intra-class phenological differences. Whether at K=6 or K=10, the hierarchical reclustering successfully unmixes the target group into distinct, fine-grained sub-species variants that align cleanly with individual field boundaries.


# Data Prepration Script

In [2]:
%pip install rasterio numpy snappy pandas geopandas rasterstats scikit-image

In [3]:
import os
import glob
import time
import warnings
import subprocess
import concurrent.futures
import multiprocessing
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.ndimage import uniform_filter, sobel, laplace, maximum_filter, minimum_filter, gaussian_filter
from skimage.filters.rank import entropy as rank_entropy
from skimage.feature import local_binary_pattern
from osgeo import gdal

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Local Processing Workflow (SNAP & QGIS)

## Step 1: Terrain Correction & Calibration (SNAP)

Perform radiometric calibration followed by terrain correction to generate georeferenced SAR backscatter images.

![Capella Processed Image](https://i.ibb.co/9mmNwYFS/image.png)

## Step 2: Image Alignment (OSGeo4W Shell | QGIS)

Align all processed SAR images onto a common spatial grid to ensure pixel-level consistency across the time series.

In [4]:
def OsGeoCode():
    BASE_DIR = Path(r"C:\Users\Akash_Chaudhari\Desktop\Data_PreProcessing")
    PROCESSED_DIR = BASE_DIR / "02_Processed_TC"
    ALIGNED_DIR = BASE_DIR / "03_Aligned_Stack"
    ALIGNED_DIR.mkdir(parents=True, exist_ok=True)
    
    gdal.UseExceptions()

    tif_files = sorted(PROCESSED_DIR.rglob("*_Cal_TC.tif"))
    if not tif_files:
        print(f"No TIFs found in {PROCESSED_DIR}!")
        return

    # Extract Master Reference
    master_tif = tif_files[0]
    print(f"--- MASTER REFERENCE ---")
    print(f"File: {master_tif.name}")

    master_ds = gdal.Open(str(master_tif))
    geo_t = master_ds.GetGeoTransform()
    x_size = master_ds.RasterXSize
    y_size = master_ds.RasterYSize
    proj = master_ds.GetProjection()

    # Calculate geographic bounding box
    minX = geo_t[0]
    maxY = geo_t[3]
    maxX = minX + geo_t[1] * x_size
    minY = maxY + geo_t[5] * y_size
    output_bounds = (minX, minY, maxX, maxY)

    print(f"Grid: {x_size} x {y_size}\n")

    # Align files
    print("--- ALIGNING ALL IMAGES ---")
    for tif in tif_files:
        out_path = ALIGNED_DIR / f"{tif.stem}_aligned.tif"
        print(f"\nProcessing: {tif.name}")

        # --- THE FIX FOR OLDER PCs ---
        warp_opts = gdal.WarpOptions(
            format="GTiff",
            width=x_size,
            height=y_size,
            outputBounds=output_bounds,
            dstSRS=proj,
            resampleAlg=gdal.GRA_Bilinear,
            
            # 1. Force GDAL to only use 512 MB of RAM to prevent crashing
            warpMemoryLimit=512, 
            
            # 2. Add a live progress bar so you know it isn't frozen
            callback=gdal.TermProgress_nocb, 
            
            creationOptions=[
                "COMPRESS=LZW", 
                "TILED=YES",
                "BIGTIFF=IF_SAFER"
            ]
        )

        gdal.Warp(str(out_path), str(tif), options=warp_opts)
        print(f"Saved: {out_path.name}")

## Step 3: NoData Removal

Remove all NoData pixels from the SAR images to retain only valid observations for subsequent processing.

In [5]:
def NoData_Removal():
    BASE_DIR = Path(r"C:\Users\Akash_Chaudhari\Desktop\Data_PreProcessing")
    INPUT_DIR = BASE_DIR / "03_Aligned_Stack"
    OUTPUT_DIR = BASE_DIR / "03b_Aligned_Masked"
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Standardized NoData value
    NEW_NODATA = -9999.0

    tif_files = sorted(INPUT_DIR.glob("*_aligned.tif"))
    if not tif_files:
        print(f"❌ No TIFs found in {INPUT_DIR}")
        return

    print("--- STARTING ZERO-PADDING MASKING ---")

    for tif in tif_files:
        out_name = tif.stem + "_Clean.tif"
        out_path = OUTPUT_DIR / out_name
        print(f"\nProcessing: {tif.name}")

        with rasterio.open(tif) as src:
            profile = src.profile
            orig_nodata = src.nodata

            # Optimize the output for SNAP and ML
            profile.update(
                nodata=NEW_NODATA,
                compress='lzw',
                tiled=True,
                blockxsize=512,
                blockysize=512
            )

            with rasterio.open(out_path, 'w', **profile) as dst:
                
                windows = list(src.block_windows(1))
                total_blocks = len(windows)
                
                # RAM-safe block processing
                for i, (block_index, window) in enumerate(windows):
                    block = src.read(1, window=window)
                    
                    # 1. Target the exact 0.0 padding generated by QGIS
                    bad_mask = (block == 0.0)
                    
                    # 2. Catch any original NoData if it exists
                    if orig_nodata is not None:
                        bad_mask |= (block == orig_nodata)

                    # 3. Apply the clean NoData value
                    block[bad_mask] = NEW_NODATA

                    dst.write(block, 1, window=window)

                    # Print progress
                    if (i + 1) % 500 == 0 or (i + 1) == total_blocks:
                        print(f"  -> Processed {i + 1}/{total_blocks} blocks ({(i+1)/total_blocks*100:.1f}%)")

        print(f"✅ Saved to: {out_path.name}")

# Step 4: Conversion to dB (QGIS GPT)

Convert all SAR backscatter values from their raw (linear) scale to the decibel (dB) scale for analysis and feature extraction.

In [6]:
def Convert_Value_DB():
    BASE_DIR = Path(r"C:\Users\Akash_Chaudhari\Desktop\Data_PreProcessing") 
    INPUT_DIR = BASE_DIR / "03b_Aligned_Masked"
    OUTPUT_DIR = BASE_DIR / "04_Filtered_dB"
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    GPT_PATH = r"C:\Program Files\esa-snap\bin\gpt.exe"
    if not os.path.exists(GPT_PATH):
        print(f"Error: Could not find SNAP GPT at {GPT_PATH}.")
        print("Please check where SNAP is installed on your PC.")
        return
    
    graph_path = r"C:\Users\Akash_Chaudhari\Desktop\Data_PreProcessing\graphs\speckle_filter.xml"

    tif_files = sorted(INPUT_DIR.glob("*_aligned_Clean.tif"))
    if not tif_files:
        print(f"No files found in {INPUT_DIR}")
        return

    print("--- STARTING SNAP NATIVE PROCESSING ---")
    
    for tif in tif_files:
        out_name = tif.stem.replace("_aligned_Clean", "_RefinedLee_dB.tif")
        out_path = OUTPUT_DIR / out_name
        
        print(f"\nProcessing: {tif.name}")
        
        # Build the command to run SNAP
        # -c 2G forces SNAP to only use 2GB of RAM cache (protects your 8GB PC)
        # -q 4 tells it to use 4 CPU threads
        cmd = [
            GPT_PATH, str(graph_path),
            f"-PsourceFile={str(tif)}",
            f"-PtargetFile={str(out_path)}",
            "-c", "4G", 
            "-q", "4"
        ]
        
        
        subprocess.run(cmd, check=True)

## Downloading the Processed Data from the Local Environment

In [7]:
!gdown 1MLetTg20egl1zw9O3rtW6N9ssmpziJzx

!unzip 04_Filtered_dB.zip

!rm 04_Filtered_dB.zip

Downloading...
From (original): https://drive.google.com/uc?id=1MLetTg20egl1zw9O3rtW6N9ssmpziJzx
From (redirected): https://drive.google.com/uc?id=1MLetTg20egl1zw9O3rtW6N9ssmpziJzx&confirm=t&uuid=e183ee4b-dd7d-4226-9493-84b24e50a438
To: /kaggle/working/04_Filtered_dB.zip
100%|██████████████████████████████████████| 1.56G/1.56G [00:19<00:00, 78.5MB/s]
Archive:  04_Filtered_dB.zip
  inflating: 04_Filtered_dB/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506_Cal_TC_RefinedLee_dB.tif  
  inflating: 04_Filtered_dB/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506_Cal_TC_RefinedLee_dB.tif.aux.xml  
  inflating: 04_Filtered_dB/CAPELLA_C14_SM_SLC_HH_20250619021410_20250619021415_Cal_TC_RefinedLee_dB.tif  
  inflating: 04_Filtered_dB/CAPELLA_C14_SM_SLC_HH_20250619021410_20250619021415_Cal_TC_RefinedLee_dB.tif.aux.xml  
  inflating: 04_Filtered_dB/CAPELLA_C14_SM_SLC_HH_20250814031124_20250814031129_Cal_TC_RefinedLee_dB.tif  
  inflating: 04_Filtered_dB/CAPELLA_C14_SM_SLC_HH_20250814031124_20

## Step 5: Calculation of Temporal Statistics

Compute temporal statistics for each pixel across all SAR acquisition dates, including **Minimum, Maximum, Mean, Standard Deviation, Valid Pixel Count,** and **Range**.

In [31]:
def Tempo_Stats():
    BASE_DIR = Path(r".")
    INPUT_DIR = BASE_DIR / "04_Filtered_dB"
    OUTPUT_DIR = BASE_DIR / "05_Temporal_Stats"
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    DB_VALID_MIN = -35.0
    DB_VALID_MAX = 5.0
    MIN_VALID_FRACTION = 0.5
    BLOCK_SIZE = 6144

    tif_files = sorted(INPUT_DIR.glob("*_RefinedLee_dB.tif"))
    if not tif_files:
        print(f"No TIFs found in {INPUT_DIR}")
        return

    n_dates = len(tif_files)
    min_valid_count = max(1, int(np.ceil(n_dates * MIN_VALID_FRACTION)))

    print(f"--- CALCULATING TEMPORAL STATS (RAM-SAFE) ACROSS {n_dates} DATES ---")
    
    src_datasets = [rasterio.open(tif) for tif in tif_files]
    master_src = src_datasets[0]
    
    full_W, full_H = master_src.width, master_src.height

    NODATA = master_src.nodata if master_src.nodata is not None else -9999.0

    master_profile = master_src.profile
    master_profile.update(dtype=rasterio.float32, nodata=NODATA, compress="deflate", zlevel=9, predictor=3, tiled=True, blockxsize=512, blockysize=512)

    stat_names = ["mean", "std", "min", "max", "range"]
    dst_datasets = {}

    for stat in stat_names:
        out_path = OUTPUT_DIR / f"Temporal_{stat.capitalize()}.tif"
        dst_datasets[stat] = rasterio.open(out_path, 'w', **master_profile)

    count_profile = master_profile.copy()
    count_profile.update(dtype=rasterio.uint8, nodata=0, predictor=2)
    count_path = OUTPUT_DIR / "Temporal_ValidCount.tif"
    dst_count = rasterio.open(count_path, 'w', **count_profile)

    print("\n🚀 Processing in large blocks to stay within 30GB RAM...")
    
    for row_off in range(0, full_H, BLOCK_SIZE):
        for col_off in range(0, full_W, BLOCK_SIZE):
            width = min(BLOCK_SIZE, full_W - col_off)
            height = min(BLOCK_SIZE, full_H - row_off)
            window = Window(col_off, row_off, width, height)
            
            blocks = [src.read(1, window=window) for src in src_datasets]
            stack = np.stack(blocks, axis=0)

            invalid = (
                (stack == NODATA)
                | ~np.isfinite(stack)
                | (stack < DB_VALID_MIN)
                | (stack > DB_VALID_MAX)
            )
            stack = np.where(invalid, np.nan, stack)

            valid_count = np.sum(~invalid, axis=0).astype(np.uint8)
            insufficient = valid_count < min_valid_count

            linear_stack = 10.0 ** (stack / 10.0)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                linear_mean = np.nanmean(linear_stack, axis=0)
                t_mean = np.where(linear_mean > 0, 10.0 * np.log10(linear_mean), np.nan)
                t_std = np.nanstd(linear_stack, axis=0)
                t_min = np.nanmin(stack, axis=0)
                t_max = np.nanmax(stack, axis=0)
                t_range = t_max - t_min

            for arr in (t_mean, t_std, t_min, t_max, t_range):
                arr[insufficient] = np.nan
            
            results = {
                "mean": np.nan_to_num(t_mean, nan=NODATA).astype(np.float32),
                "std": np.nan_to_num(t_std, nan=NODATA).astype(np.float32),
                "min": np.nan_to_num(t_min, nan=NODATA).astype(np.float32),
                "max": np.nan_to_num(t_max, nan=NODATA).astype(np.float32),
                "range": np.nan_to_num(t_range, nan=NODATA).astype(np.float32),
            }

            for stat in stat_names:
                dst_datasets[stat].write(results[stat], 1, window=window)

            dst_count.write(valid_count, 1, window=window)

    for dst in dst_datasets.values():
        dst.close()
    dst_count.close()
    
    for src in src_datasets:
        src.close()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## Step 6: Texture Feature Extraction

Extract spatial texture features from each SAR image using a moving window. The generated features include **Local Variance, Coefficient of Variation (CoV), Gradient Magnitude, Laplacian, Local Range,** and **Entropy**, providing information about surface heterogeneity and structural patterns.

In [ ]:
def process_block(args):
    import os
    import numpy as np
    import rasterio
    from rasterio.windows import Window
    from scipy.ndimage import uniform_filter, sobel, laplace, maximum_filter, minimum_filter
    from skimage.filters.rank import entropy as rank_entropy
    import multiprocessing

    INPUT_DIR = "04_Filtered_dB"
    OUTPUT_DIR = "06_Fast_Texture"
    NODATA_VAL = -9999.0

    WINDOW_SIZE = 7
    PAD_SIZE = WINDOW_SIZE // 2
    BLOCK_SIZE = 4096
    CORES = multiprocessing.cpu_count()

    FEATURES = ['Variance', 'CoV', 'Gradient', 'Laplacian', 'Range', 'Entropy']

    filepath, col_off, row_off, width, height, full_W, full_H, global_vmin, global_vmax = args

    read_col_off = max(0, col_off - PAD_SIZE)
    read_row_off = max(0, row_off - PAD_SIZE)
    read_width = min(full_W, col_off + width + PAD_SIZE) - read_col_off
    read_height = min(full_H, row_off + height + PAD_SIZE) - read_row_off

    with rasterio.open(filepath) as src:
        img_block = src.read(1, window=Window(read_col_off, read_row_off, read_width, read_height))

    valid_mask = (img_block != NODATA_VAL) & np.isfinite(img_block)

    if not np.any(valid_mask):
        empty = np.full((height, width), NODATA_VAL, dtype=np.float32)
        return (col_off, row_off, width, height, {f: empty for f in FEATURES})

    mask_float = valid_mask.astype(np.float32)
    mean_mask = uniform_filter(mask_float, size=WINDOW_SIZE, mode='constant', cval=0.0)
    safe_mask = np.where(mean_mask > 0, mean_mask, 1.0)

    safe_data_db = np.where(valid_mask, img_block, 0.0).astype(np.float32)

    mean_db = uniform_filter(safe_data_db, size=WINDOW_SIZE, mode='constant', cval=0.0) / safe_mask
    mean_sq_db = uniform_filter(safe_data_db**2, size=WINDOW_SIZE, mode='constant', cval=0.0) / safe_mask
    local_var_db = np.maximum(mean_sq_db - (mean_db**2), 0)

    safe_data_lin = np.where(valid_mask, 10.0 ** (img_block / 10.0), 0.0).astype(np.float32)

    mean_lin = uniform_filter(safe_data_lin, size=WINDOW_SIZE, mode='constant', cval=0.0) / safe_mask
    mean_sq_lin = uniform_filter(safe_data_lin**2, size=WINDOW_SIZE, mode='constant', cval=0.0) / safe_mask
    local_var_lin = np.maximum(mean_sq_lin - (mean_lin**2), 0)
    local_std_lin = np.sqrt(local_var_lin)
    cv = local_std_lin / (mean_lin + 1e-15)

    sobel_x = sobel(safe_data_db, axis=1, mode='reflect')
    sobel_y = sobel(safe_data_db, axis=0, mode='reflect')
    gradient = np.hypot(sobel_x, sobel_y)
    laplacian = laplace(safe_data_db, mode='reflect')

    strict_3x3_mask = minimum_filter(valid_mask, size=3)
    gradient[~strict_3x3_mask] = NODATA_VAL
    laplacian[~strict_3x3_mask] = NODATA_VAL

    max_safe = np.where(valid_mask, img_block, -np.inf)
    min_safe = np.where(valid_mask, img_block, np.inf)

    local_max = maximum_filter(max_safe, size=WINDOW_SIZE, mode='reflect')
    local_min = minimum_filter(min_safe, size=WINDOW_SIZE, mode='reflect')
    local_range = np.maximum(local_max - local_min, 0)

    scale = max(global_vmax - global_vmin, 1e-6)
    normalized = (img_block - global_vmin) / scale
    quantized = np.clip(normalized * 63, 0, 63).astype(np.uint8)

    footprint = np.ones((WINDOW_SIZE, WINDOW_SIZE), dtype=np.uint8)
    entropy_arr = rank_entropy(quantized, footprint).astype(np.float32)

    outputs = {
        'Variance': local_var_db,
        'CoV': cv,
        'Gradient': gradient,
        'Laplacian': laplacian,
        'Range': local_range,
        'Entropy': entropy_arr
    }

    pad_top = row_off - read_row_off
    pad_left = col_off - read_col_off

    final_outputs = {}
    for feat, data in outputs.items():
        if feat not in ['Gradient', 'Laplacian']:
            data[~valid_mask] = NODATA_VAL
        final_outputs[feat] = data[pad_top: pad_top + height, pad_left: pad_left + width]

    return (col_off, row_off, width, height, final_outputs)


def process_file(filepath):
    import os
    import glob
    import time
    import numpy as np
    import rasterio
    from rasterio.windows import Window
    import concurrent.futures
    import multiprocessing

    INPUT_DIR = "04_Filtered_dB"
    OUTPUT_DIR = "06_Fast_Texture"
    NODATA_VAL = -9999.0

    WINDOW_SIZE = 7
    PAD_SIZE = WINDOW_SIZE // 2
    BLOCK_SIZE = 6144
    CORES = multiprocessing.cpu_count()

    FEATURES = ['Variance', 'CoV', 'Gradient', 'Laplacian', 'Range', 'Entropy']

    basename = os.path.basename(filepath)
    date_str = basename.split('_')[5][:8]
    print(f"\n🚀 Processing Date: {date_str} on {CORES} cores (Block Size: {BLOCK_SIZE})...")
    start_time = time.time()

    with rasterio.open(filepath) as src:
        meta = src.meta.copy()
        full_W, full_H = src.width, src.height

        print("   -> Computing global bounds for Entropy quantization...")
        full_img = src.read(1)
        valid_pixels = full_img[(full_img != NODATA_VAL) & np.isfinite(full_img)]

        if valid_pixels.size == 0:
            print("   -> Image is entirely NoData. Skipping.")
            return

        global_vmin, global_vmax = np.percentile(valid_pixels, [2, 98])

        del full_img
        del valid_pixels

    meta.update(
        dtype=rasterio.float32,
        nodata=NODATA_VAL,
        compress='deflate', zlevel=9, predictor=3,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )

    out_files = {}
    for feat in FEATURES:
        out_path = os.path.join(OUTPUT_DIR, f"{feat}_{date_str}.tif")
        out_files[feat] = rasterio.open(out_path, 'w', **meta)

    tasks = []
    for row_off in range(0, full_H, BLOCK_SIZE):
        for col_off in range(0, full_W, BLOCK_SIZE):
            width = min(BLOCK_SIZE, full_W - col_off)
            height = min(BLOCK_SIZE, full_H - row_off)
            tasks.append((filepath, col_off, row_off, width, height, full_W, full_H, global_vmin, global_vmax))

    print(f"   -> Distributed into {len(tasks)} tasks.")

    with concurrent.futures.ProcessPoolExecutor(max_workers=CORES) as executor:
        for result in executor.map(process_block, tasks):
            col_off, row_off, width, height, feat_data_dict = result
            write_window = Window(col_off, row_off, width, height)

            for feat in FEATURES:
                out_files[feat].write(feat_data_dict[feat], 1, window=write_window)

    for handle in out_files.values():
        handle.close()

    print(f"   ✅ Finished {date_str} in {time.time() - start_time:.2f} seconds.")


def calculate_textures():
    import os
    import glob

    INPUT_DIR = "04_Filtered_dB"
    input_files = glob.glob(os.path.join(INPUT_DIR, "*.tif"))

    if not input_files:
        print(f"No TIFF files found in {INPUT_DIR}.")
    else:
        for f in input_files:
            process_file(f)
        print("\n🎉 HPC Texture Extraction Complete!")

In [24]:
def process_texture_file(filepath):
    basename = os.path.basename(filepath)
    date_str = basename.split('_')[5][:8]
    print(f"\n🚀 Processing Date: {date_str} on {CORES} cores (Block Size: {BLOCK_SIZE})...")
    start_time = time.time()
    
    with rasterio.open(filepath) as src:
        meta = src.meta.copy()
        full_W, full_H = src.width, src.height
        
        print("   -> Computing global bounds for Entropy quantization...")
        # Read the full image temporarily just for bounds calculation
        full_img = src.read(1)
        valid_pixels = full_img[(full_img != NODATA_VAL) & np.isfinite(full_img)]
        
        if valid_pixels.size == 0:
            print("   -> Image is entirely NoData. Skipping.")
            return
            
        # Use robust 2nd and 98th percentiles to avoid extreme scatterer distortion
        global_vmin, global_vmax = np.percentile(valid_pixels, [2, 98])
        
        # Free memory before launching workers
        del full_img
        del valid_pixels
        
    meta.update(
        dtype=rasterio.float32, 
        nodata=NODATA_VAL,
        compress='deflate', zlevel=9, predictor=3,
        tiled=True,
        blockxsize=512,
        blockysize=512
    )
    
    out_files = {}
    for feat in FEATURES:
        out_path = os.path.join(OUTPUT_DIR, f"{feat}_{date_str}.tif")
        out_files[feat] = rasterio.open(out_path, 'w', **meta)
        
    tasks = []
    for row_off in range(0, full_H, BLOCK_SIZE):
        for col_off in range(0, full_W, BLOCK_SIZE):
            width = min(BLOCK_SIZE, full_W - col_off)
            height = min(BLOCK_SIZE, full_H - row_off)
            tasks.append((filepath, col_off, row_off, width, height, full_W, full_H, global_vmin, global_vmax))
            
    print(f"   -> Distributed into {len(tasks)} tasks.")
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=CORES) as executor:
        for result in executor.map(process_block, tasks):
            col_off, row_off, width, height, feat_data_dict = result
            write_window = Window(col_off, row_off, width, height)
            
            for feat in FEATURES:
                out_files[feat].write(feat_data_dict[feat], 1, window=write_window)
            
    for handle in out_files.values():
        handle.close()
        
    print(f"   ✅ Finished {date_str} in {time.time() - start_time:.2f} seconds.")

## Step 7: DEM Preparation

Download the DEM, reproject it to UTM, and align it to the SAR grid.


In [11]:
# Downloading DEM
!gdown 1I9_qyayzYu4I11-s17_M9liD0bevaOBR

Downloading...
From: https://drive.google.com/uc?id=1I9_qyayzYu4I11-s17_M9liD0bevaOBR
To: /kaggle/working/AOI_DEM_30m.tif
100%|███████████████████████████████████████| 2.74M/2.74M [00:00<00:00, 135MB/s]


In [25]:
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_DEM = "./AOI_DEM_30m.tif"
OUTPUT_DEM = "./DEM_UTM43N.tif"
TARGET_CRS = 'EPSG:32643'  # UTM Zone 43N (Meters)
NODATA_VAL = -9999.0

def reproject_dem():
    print("🚀 Reprojecting DEM to Projected Coordinate System (UTM)...")
    
    with rasterio.open(INPUT_DEM) as src:
        # Calculate the new transform and dimensions in meters
        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )
        
        # Copy metadata and update for the new CRS
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': TARGET_CRS,
            'transform': transform,
            'width': width,
            'height': height,
            'nodata': NODATA_VAL,
            'dtype': rasterio.float32,
            'compress': 'lzw'
        })

        with rasterio.open(OUTPUT_DEM, 'w', **kwargs) as dst:
            print(f"   -> Target CRS: {TARGET_CRS}")
            print(f"   -> New Dimensions: {width} x {height}")
            
            for i in range(1, src.count + 1):
                # Read original data
                source_data = src.read(i)
                
                # Treat NaN or extremely low anomalous values as NoData if they exist
                source_data = np.where(np.isnan(source_data), NODATA_VAL, source_data)
                
                reproject(
                    source=source_data,
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata, # Original has None
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    dst_nodata=NODATA_VAL,
                    resampling=Resampling.bilinear # Crucial for elevation data
                )
                
    print("✅ Reprojection complete! Update INPUT_DEM in the terrain script to point to DEM_UTM43N.tif")

## Step 8: Terrain Features

Extract elevation, slope, aspect, and curvature from the DEM.


In [26]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from scipy.ndimage import sobel, laplace, minimum_filter
import concurrent.futures
import multiprocessing
import time
import glob
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

INPUT_DEM = "./DEM_UTM43N.tif"
OUTPUT_DIR = "07_Terrain_Features"
NODATA_VAL = -9999.0
WINDOW_SIZE = 3
PAD_SIZE = 1
BLOCK_SIZE = 4096
CORES = multiprocessing.cpu_count()

os.makedirs(OUTPUT_DIR, exist_ok=True)
FEATURES = ['Elevation', 'Slope', 'Aspect', 'Laplacian_Curvature']

def process_dem_block(args):
    filepath, col_off, row_off, width, height, full_W, full_H, res_x, res_y = args
    read_col_off = max(0, col_off - PAD_SIZE)
    read_row_off = max(0, row_off - PAD_SIZE)
    read_width = min(full_W, col_off + width + PAD_SIZE) - read_col_off
    read_height = min(full_H, row_off + height + PAD_SIZE) - read_row_off
    
    with rasterio.open(filepath) as src:
        img_block = src.read(1, window=Window(read_col_off, read_row_off, read_width, read_height))
    
    valid_mask = (img_block != NODATA_VAL) & np.isfinite(img_block)
    if not np.any(valid_mask):
        empty = np.full((height, width), NODATA_VAL, dtype=np.float32)
        return (col_off, row_off, width, height, {f: empty for f in FEATURES})
    
    block_mean = np.nanmean(np.where(valid_mask, img_block, np.nan))
    safe_dem = np.where(valid_mask, img_block, block_mean)
    
    px_x = abs(res_x)
    px_y = abs(res_y)
    
    dz_dx = sobel(safe_dem, axis=1, mode='reflect') / (8.0 * px_x)
    dz_dy = sobel(safe_dem, axis=0, mode='reflect') / (8.0 * px_y)
    
    slope_deg = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype(np.float32)
    
    aspect_deg = np.degrees(np.arctan2(dz_dy, -dz_dx)).astype(np.float32)
    aspect_deg = np.where(aspect_deg < 0, aspect_deg + 360.0, aspect_deg)
    aspect_deg = np.where(aspect_deg == 360.0, 0.0, aspect_deg)
    aspect_deg[slope_deg < 0.1] = -1.0 
    
    laplacian_curvature = laplace(safe_dem, mode='reflect').astype(np.float32)
    
    strict_3x3_mask = minimum_filter(valid_mask, size=3)
    outputs = {
        'Elevation': img_block.astype(np.float32),
        'Slope': slope_deg,
        'Aspect': aspect_deg,
        'Laplacian_Curvature': laplacian_curvature
    }
    
    pad_top = row_off - read_row_off
    pad_left = col_off - read_col_off
    
    final_outputs = {}
    for feat, data in outputs.items():
        if feat == 'Elevation':
            data[~valid_mask] = NODATA_VAL
        else:
            data[~strict_3x3_mask] = NODATA_VAL
        final_outputs[feat] = data[pad_top : pad_top + height, pad_left : pad_left + width]
        
    return (col_off, row_off, width, height, final_outputs)

def process_dem(filepath):
    print(f"\n🚀 Step 1: Computing Terrain on {CORES} cores...")
    start_time = time.time()
    
    with rasterio.open(filepath) as src:
        meta = src.meta.copy()
        full_W, full_H = src.width, src.height
        res_x, res_y = src.res[0], src.res[1]

    meta.update(dtype=rasterio.float32, nodata=NODATA_VAL, compress='deflate', zlevel=9, predictor=3, tiled=True, blockxsize=256, blockysize=256)
    
    temp_dir = "/dev/shm/temp_terrain"
    os.makedirs(temp_dir, exist_ok=True)
    
    out_files = {}
    for feat in FEATURES:
        out_files[feat] = rasterio.open(os.path.join(temp_dir, f"{feat}_utm.tif"), 'w', **meta)
        
    tasks = []
    for row_off in range(0, full_H, BLOCK_SIZE):
        for col_off in range(0, full_W, BLOCK_SIZE):
            tasks.append((filepath, col_off, row_off, min(BLOCK_SIZE, full_W - col_off), min(BLOCK_SIZE, full_H - row_off), full_W, full_H, res_x, res_y))
            
    with concurrent.futures.ProcessPoolExecutor(max_workers=CORES) as executor:
        for result in executor.map(process_dem_block, tasks):
            col_off, row_off, width, height, feat_data = result
            for feat in FEATURES:
                out_files[feat].write(feat_data[feat], 1, window=Window(col_off, row_off, width, height))
            
    for handle in out_files.values():
        handle.close()
    print(f"   ✅ UTM computation finished in {time.time() - start_time:.2f}s.")
    
    print("\n🚀 Step 2: Reprojecting and Strictly Masking to SAR Boundaries...")
    sar_ref_path = glob.glob("04_Filtered_dB/*.tif")[0]
    with rasterio.open(sar_ref_path) as ref:
        ref_transform = ref.transform
        ref_crs = ref.crs
        ref_width = ref.width
        ref_height = ref.height
        
        print("   -> Extracting master SAR boundary mask...")
        sar_data = ref.read(1)
        sar_valid_mask = (sar_data != NODATA_VAL) & np.isfinite(sar_data)
        del sar_data  # Free massive array
        
    out_meta = ref.meta.copy()
    out_meta.update(dtype=rasterio.float32, nodata=NODATA_VAL, compress='deflate', zlevel=9, predictor=3, tiled=True, blockxsize=512, blockysize=512, count=1)
    
    for feat in FEATURES:
        utm_path = os.path.join(temp_dir, f"{feat}_utm.tif")
        final_path = os.path.join(OUTPUT_DIR, f"{feat}.tif")
        print(f"   -> Aligning and Masking {feat}...", end="", flush=True)
        t0 = time.time()
        
        with rasterio.open(utm_path) as src:
            with rasterio.open(final_path, 'w', **out_meta) as dst:
                source_data = src.read(1)
                source_data = np.where(np.isnan(source_data), NODATA_VAL, source_data)
                
                dest_array = np.full((ref_height, ref_width), NODATA_VAL, dtype=np.float32)
                
                reproject(
                    source=source_data,
                    destination=dest_array,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=NODATA_VAL,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    dst_nodata=NODATA_VAL,
                    resampling=Resampling.bilinear
                )
                
                # Apply the SAR footprint mask strictly
                dest_array[~sar_valid_mask] = NODATA_VAL
                
                dst.write(dest_array, 1)
                del dest_array
        
        os.remove(utm_path)
        print(f" Done! ({time.time() - t0:.1f}s)")
        


## Step 9: Temporal Dynamics

Compute pairwise temporal change features between SAR dates.


In [27]:
import os
import glob
import numpy as np
import rasterio
from rasterio.windows import Window
import concurrent.futures
import multiprocessing
import time
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ==========================================
# HPC CONFIGURATION
# ==========================================
INPUT_DIR = "04_Filtered_dB"
OUTPUT_DIR = "08_Temporal_Dynamics"
NODATA_VAL = -9999.0

BLOCK_SIZE = 4096
CORES = multiprocessing.cpu_count()

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# HELPER: Sort Files Chronologically
# ==========================================
def extract_date(filepath):
    """Extracts YYYYMMDD from Capella filenames to ensure strict chronological ordering."""
    basename = os.path.basename(filepath)
    # CAPELLA_C14_SM_SLC_HH_YYYYMMDD...
    date_str = basename.split('_')[5][:8]
    return date_str

# ==========================================
# ALGORITHM: Vectorized Pairwise Deltas
# ==========================================
def process_temporal_block(args):
    files, col_off, row_off, width, height = args
    
    # Read the exact same spatial window from all 4 dates
    window = Window(col_off, row_off, width, height)
    arrays = []
    masks = []
    
    for f in files:
        with rasterio.open(f) as src:
            arr = src.read(1, window=window)
            arrays.append(arr)
            masks.append((arr != NODATA_VAL) & np.isfinite(arr))
            
    t1, t2, t3, t4 = arrays
    m1, m2, m3, m4 = masks
    
    # Pairwise NoData masking: A delta is only valid if BOTH dates are valid
    valid_21 = m2 & m1
    valid_32 = m3 & m2
    valid_43 = m4 & m3
    valid_41 = m4 & m1
    
    # Compute differences (dB subtraction = Power Ratio)
    delta_21 = np.where(valid_21, t2 - t1, NODATA_VAL).astype(np.float32)
    delta_32 = np.where(valid_32, t3 - t2, NODATA_VAL).astype(np.float32)
    delta_43 = np.where(valid_43, t4 - t3, NODATA_VAL).astype(np.float32)
    delta_41 = np.where(valid_41, t4 - t1, NODATA_VAL).astype(np.float32)
    
    return (col_off, row_off, width, height, delta_21, delta_32, delta_43, delta_41)

# ==========================================
# MAIN PIPELINE MANAGER
# ==========================================
def process_dynamics():
    print(f"\n🚀 Processing Temporal Dynamics on {CORES} cores (Block Size: {BLOCK_SIZE})...")
    start_time = time.time()
    
    input_files = glob.glob(os.path.join(INPUT_DIR, "*.tif"))
    if len(input_files) < 4:
        print(f"❌ Error: Found {len(input_files)} files. Need exactly 4 for this hardcoded sequence.")
        return
        
    # Strictly sort files by date
    input_files = sorted(input_files, key=extract_date)
    dates = [extract_date(f) for f in input_files]
    print(f"   -> Chronological Sequence: {dates}")
    
    # Define output names based on the dates
    out_names = [
        f"Delta_{dates[1]}_minus_{dates[0]}.tif",
        f"Delta_{dates[2]}_minus_{dates[1]}.tif",
        f"Delta_{dates[3]}_minus_{dates[2]}.tif",
        f"Delta_Overall_{dates[3]}_minus_{dates[0]}.tif"
    ]
    
    # Use the first file as the spatial template
    with rasterio.open(input_files[0]) as src:
        meta = src.meta.copy()
        full_W, full_H = src.width, src.height
        
    meta.update(
        dtype=rasterio.float32, 
        nodata=NODATA_VAL,
        compress='deflate', zlevel=9, predictor=3,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )
    
    out_files = {}
    for name in out_names:
        out_path = os.path.join(OUTPUT_DIR, name)
        out_files[name] = rasterio.open(out_path, 'w', **meta)
        
    tasks = []
    for row_off in range(0, full_H, BLOCK_SIZE):
        for col_off in range(0, full_W, BLOCK_SIZE):
            width = min(BLOCK_SIZE, full_W - col_off)
            height = min(BLOCK_SIZE, full_H - row_off)
            tasks.append((input_files, col_off, row_off, width, height))
            
    print(f"   -> Distributed into {len(tasks)} tasks.")
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=CORES) as executor:
        for result in executor.map(process_temporal_block, tasks):
            col_off, row_off, width, height, d21, d32, d43, d41 = result
            write_window = Window(col_off, row_off, width, height)
            
            # Write sequentially
            out_files[out_names[0]].write(d21, 1, window=write_window)
            out_files[out_names[1]].write(d32, 1, window=write_window)
            out_files[out_names[2]].write(d43, 1, window=write_window)
            out_files[out_names[3]].write(d41, 1, window=write_window)
            
    for handle in out_files.values():
        handle.close()
        
    print(f"   ✅ Finished Temporal Dynamics in {time.time() - start_time:.2f} seconds.")



## Step 10: LBP Texture

Extract local binary pattern texture from each SAR image.


In [28]:
import os
import glob
import numpy as np
import rasterio
from rasterio.windows import Window
from skimage.feature import local_binary_pattern
from scipy.ndimage import minimum_filter
import concurrent.futures
import multiprocessing
import time
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ==========================================
# HPC CONFIGURATION
# ==========================================
INPUT_DIR = "04_Filtered_dB"
OUTPUT_DIR = "09_Fast_Texture"
NODATA_VAL = -9999.0

# LBP Parameters (Radius 3 matches a 7x7 spatial context)
LBP_RADIUS = 3
LBP_POINTS = 24
LBP_METHOD = 'uniform'

PAD_SIZE = LBP_RADIUS
BLOCK_SIZE = 4096
CORES = multiprocessing.cpu_count()

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# ALGORITHM: Vectorized LBP
# ==========================================
def process_lbp_block(args):
    filepath, col_off, row_off, width, height, full_W, full_H = args
    
    # Calculate read window with radius-based overlap padding
    read_col_off = max(0, col_off - PAD_SIZE)
    read_row_off = max(0, row_off - PAD_SIZE)
    read_width = min(full_W, col_off + width + PAD_SIZE) - read_col_off
    read_height = min(full_H, row_off + height + PAD_SIZE) - read_row_off
    
    with rasterio.open(filepath) as src:
        img_block = src.read(1, window=Window(read_col_off, read_row_off, read_width, read_height))
    
    valid_mask = (img_block != NODATA_VAL) & np.isfinite(img_block)
    
    if not np.any(valid_mask):
        empty = np.full((height, width), NODATA_VAL, dtype=np.float32)
        return (col_off, row_off, width, height, empty)
    
    # Replace NoData with the block median to prevent artificial gradients 
    # from skewing the LBP threshold comparisons near the swath edge.
    block_median = np.nanmedian(np.where(valid_mask, img_block, np.nan))
    safe_data = np.where(valid_mask, img_block, block_median)
    
    # ---------------------------------------------------------
    # Compute Uniform Local Binary Pattern
    # ---------------------------------------------------------
    lbp_array = local_binary_pattern(safe_data, LBP_POINTS, LBP_RADIUS, LBP_METHOD).astype(np.float32)
    
    # ---------------------------------------------------------
    # Strict Boundary Masking
    # ---------------------------------------------------------
    # LBP requires the entire circular radius to be valid data.
    # We use a square footprint (2*Radius + 1) to strictly remove any pixel 
    # whose LBP code was influenced by a padded boundary.
    strict_mask = minimum_filter(valid_mask, size=(2 * LBP_RADIUS + 1))
    lbp_array[~strict_mask] = NODATA_VAL
    
    # Trim the padding to match the exact writing window
    pad_top = row_off - read_row_off
    pad_left = col_off - read_col_off
    
    final_lbp = lbp_array[pad_top : pad_top + height, pad_left : pad_left + width]
        
    return (col_off, row_off, width, height, final_lbp)

# ==========================================
# MAIN PIPELINE MANAGER
# ==========================================
def process_lbp_file(filepath):
    basename = os.path.basename(filepath)
    date_str = basename.split('_')[5][:8]
    print(f"\n🚀 Processing LBP Date: {date_str} on {CORES} cores...")
    start_time = time.time()
    
    with rasterio.open(filepath) as src:
        meta = src.meta.copy()
        full_W, full_H = src.width, src.height
        
    meta.update(
        dtype=rasterio.float32, 
        nodata=NODATA_VAL,
        compress='deflate', zlevel=9, predictor=3,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )
    
    out_name = f"LBP_Uniform_{date_str}.tif"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    
    tasks = []
    for row_off in range(0, full_H, BLOCK_SIZE):
        for col_off in range(0, full_W, BLOCK_SIZE):
            width = min(BLOCK_SIZE, full_W - col_off)
            height = min(BLOCK_SIZE, full_H - row_off)
            tasks.append((filepath, col_off, row_off, width, height, full_W, full_H))
            
    print(f"   -> Distributed into {len(tasks)} tasks.")
    
    with rasterio.open(out_path, 'w', **meta) as dst:
        with concurrent.futures.ProcessPoolExecutor(max_workers=CORES) as executor:
            for result in executor.map(process_lbp_block, tasks):
                col_off, row_off, width, height, lbp_data = result
                write_window = Window(col_off, row_off, width, height)
                dst.write(lbp_data, 1, window=write_window)
            
    print(f"   ✅ Saved {out_name} in {time.time() - start_time:.2f} seconds.")

def calculate_lbp_texture():
    input_files = glob.glob(os.path.join(INPUT_DIR, "*.tif"))
    if not input_files:
        print(f"❌ No TIFF files found in {INPUT_DIR}.")
    else:
        for f in input_files:
            process_lbp_file(f)
        print("\n🎉 HPC LBP Texture Extraction Complete!")


## Step 11: Structure Tensor

Extract anisotropy and tensor energy features.


In [29]:
import os
import glob
import numpy as np
import rasterio
from rasterio.windows import Window
from scipy.ndimage import gaussian_filter, minimum_filter
import concurrent.futures
import multiprocessing
import time
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ==========================================
# HPC CONFIGURATION
# ==========================================
INPUT_DIR = "04_Filtered_dB"
OUTPUT_DIR = "10_Fast_Texture"
NODATA_VAL = -9999.0

# Structure Tensor Scales
# Inner scale: Gaussian smoothing applied during derivative calculation (suppresses speckle)
SIGMA_INNER = 1.0 
# Outer scale: Gaussian smoothing applied to integrate the tensor (captures the texture pattern)
SIGMA_OUTER = 2.0 

# A Gaussian filter effectively reaches 0 at ~3 to 4 sigma. 
# We pad by 4 * SIGMA_OUTER to be mathematically bulletproof against edge effects.
PAD_SIZE = int(np.ceil(4 * SIGMA_OUTER))
BLOCK_SIZE = 4096
CORES = max(1, multiprocessing.cpu_count())

os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURES = ['Anisotropy', 'Tensor_Energy']

# ==========================================
# ALGORITHM: Vectorized Structure Tensor
# ==========================================
def process_tensor_block(args):
    filepath, col_off, row_off, width, height, full_W, full_H = args
    
    # Calculate read window with Gaussian-safe overlap padding
    read_col_off = max(0, col_off - PAD_SIZE)
    read_row_off = max(0, row_off - PAD_SIZE)
    read_width = min(full_W, col_off + width + PAD_SIZE) - read_col_off
    read_height = min(full_H, row_off + height + PAD_SIZE) - read_row_off
    
    with rasterio.open(filepath) as src:
        img_block = src.read(1, window=Window(read_col_off, read_row_off, read_width, read_height))
    
    valid_mask = (img_block != NODATA_VAL) & np.isfinite(img_block)
    
    if not np.any(valid_mask):
        empty = np.full((height, width), NODATA_VAL, dtype=np.float32)
        return (col_off, row_off, width, height, {f: empty for f in FEATURES})
    
    # Fill NoData with block median to stabilize the gradients at the swath edges
    block_median = np.nanmedian(np.where(valid_mask, img_block, np.nan))
    safe_data = np.where(valid_mask, img_block, block_median).astype(np.float32)
    
    # ---------------------------------------------------------
    # 1. Inner Scale: Gaussian Derivatives
    # ---------------------------------------------------------
    # order=[0, 1] means derivative in X, order=[1, 0] means derivative in Y
    Ix = gaussian_filter(safe_data, sigma=SIGMA_INNER, order=[0, 1], mode='reflect')
    Iy = gaussian_filter(safe_data, sigma=SIGMA_INNER, order=[1, 0], mode='reflect')
    
    # ---------------------------------------------------------
    # 2. Outer Scale: Tensor Integration
    # ---------------------------------------------------------
    Sxx = gaussian_filter(Ix**2, sigma=SIGMA_OUTER, mode='reflect')
    Syy = gaussian_filter(Iy**2, sigma=SIGMA_OUTER, mode='reflect')
    Sxy = gaussian_filter(Ix * Iy, sigma=SIGMA_OUTER, mode='reflect')
    
    # ---------------------------------------------------------
    # 3. Eigenvalue Analysis
    # ---------------------------------------------------------
    trace = Sxx + Syy
    det = (Sxx * Syy) - (Sxy**2)
    
    # Calculate eigenvalues: (Trace/2) +/- sqrt((Trace/2)^2 - det)
    # Use np.maximum to strictly prevent negative values inside sqrt due to float precision limits
    diff = np.sqrt(np.maximum((trace / 2.0)**2 - det, 0.0))
    
    lambda1 = (trace / 2.0) + diff  # Primary structure strength
    lambda2 = (trace / 2.0) - diff  # Secondary structure strength
    
    # ---------------------------------------------------------
    # 4. Feature Extraction
    # ---------------------------------------------------------
    # Anisotropy (Coherence): 0 = Isotropic (chaotic), 1 = Linear (crop row)
    anisotropy = (lambda1 - lambda2) / (trace + 1e-8)
    
    # Tensor Energy: Overall structural magnitude in the region
    energy = trace
    
    outputs = {
        'Anisotropy': anisotropy.astype(np.float32),
        'Tensor_Energy': energy.astype(np.float32)
    }
    
    # ---------------------------------------------------------
    # Strict Boundary Masking
    # ---------------------------------------------------------
    # The integration scale smears data over ~ 2*PAD_SIZE. We rigorously mask out 
    # any pixel whose eigenvalues were mathematically touched by the NoData boundary.
    strict_mask = minimum_filter(valid_mask, size=(2 * PAD_SIZE + 1))
    
    pad_top = row_off - read_row_off
    pad_left = col_off - read_col_off
    
    final_outputs = {}
    for feat, data in outputs.items():
        data[~strict_mask] = NODATA_VAL
        final_outputs[feat] = data[pad_top : pad_top + height, pad_left : pad_left + width]
        
    return (col_off, row_off, width, height, final_outputs)

# ==========================================
# MAIN PIPELINE MANAGER
# ==========================================
def process_tensor_file(filepath):
    basename = os.path.basename(filepath)
    date_str = basename.split('_')[5][:8]
    print(f"\n🚀 Processing Structure Tensor Date: {date_str} on {CORES} cores...")
    start_time = time.time()
    
    with rasterio.open(filepath) as src:
        meta = src.meta.copy()
        full_W, full_H = src.width, src.height
        
    meta.update(
        dtype=rasterio.float32, 
        nodata=NODATA_VAL,
        compress='deflate', zlevel=9, predictor=3,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )
    
    out_files = {}
    for feat in FEATURES:
        out_path = os.path.join(OUTPUT_DIR, f"{feat}_{date_str}.tif")
        out_files[feat] = rasterio.open(out_path, 'w', **meta)
        
    tasks = []
    for row_off in range(0, full_H, BLOCK_SIZE):
        for col_off in range(0, full_W, BLOCK_SIZE):
            width = min(BLOCK_SIZE, full_W - col_off)
            height = min(BLOCK_SIZE, full_H - row_off)
            tasks.append((filepath, col_off, row_off, width, height, full_W, full_H))
            
    print(f"   -> Distributed into {len(tasks)} tasks.")
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=CORES) as executor:
        for result in executor.map(process_tensor_block, tasks):
            col_off, row_off, width, height, feat_data_dict = result
            write_window = Window(col_off, row_off, width, height)
            
            for feat in FEATURES:
                out_files[feat].write(feat_data_dict[feat], 1, window=write_window)
            
    for handle in out_files.values():
        handle.close()
        
    print(f"   ✅ Finished {date_str} in {time.time() - start_time:.2f} seconds.")

def calculate_structure_tensor():
    input_files = glob.glob(os.path.join(INPUT_DIR, "*.tif"))
    if not input_files:
        print(f"❌ No TIFF files found in {INPUT_DIR}.")
    else:
        for f in input_files:
            process_tensor_file(f)
        print("\n🎉 HPC Structure Tensor Extraction Complete!")


## Final Run

Run the full pipeline in one place.


In [ ]:
def main():
    # # Local Processing
    
    # OsGeoCode()
    # NoData_Removal()
    # Convert_Value_DB()
    
    # # Data Pipeline
    Tempo_Stats()
    calculate_textures()
    reproject_dem()
    process_dem(INPUT_DEM)
    process_dynamics()
    calculate_lbp_texture()
    calculate_structure_tensor()

main()


🚀 Processing Date: 20250814 on 4 cores (Block Size: 4096)...
   -> Computing global bounds for Entropy quantization...
   -> Distributed into 25 tasks.


# Training Script

In [ ]:
ROOT = Path(".")

FILES_TO_DELETE = [
    "05_Temporal_Stats/Temporal_Max.tif",
    "06_Fast_Texture/Range_20250606.tif",
    "06_Fast_Texture/Range_20250814.tif",
    "06_Fast_Texture/Range_20251013.tif",
    "10_Fast_Texture/Tensor_Energy_20250606.tif",
    "10_Fast_Texture/Tensor_Energy_20250619.tif",
    "10_Fast_Texture/Tensor_Energy_20250814.tif",
    "10_Fast_Texture/Tensor_Energy_20251013.tif",
]

for file in FILES_TO_DELETE:
    path = ROOT / file
    if path.exists():
        path.unlink()
        print(f"Deleted: {path}")


## Step 1: Expert-in-the-Loop Agricultural Masking
**Objective**: Constrain the statistical distributions strictly to confirmed vegetated croplands.
**Methodology**: 
Prior to this notebook, an initial baseline clustering was generated. Analysts manually overlaid these clusters with high-resolution imagery and **manually extracted the exact cluster** that represented the crop areas. 

This manually verified mask (`Abhay Best.tif`) is now projected onto our native SAR feature space using nearest-neighbor resampling. This strictly isolates the confirmed crop pixels, enabling our subsequent PCA and GMM algorithms to focus entirely on separating *different crop types*, rather than separating crops from non-crops.


In [ ]:
# crop mask
!gdown 1uxKFEgkfRRFNWjFnvlU0WnnuvzWEDeF8
!unzip -o "Abhay Best.zip"
!rm 'Abhay Best.zip'

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import glob
from pathlib import Path

# Reference raster
ref_candidates = glob.glob("04_Filtered_dB/*.tif")
if ref_candidates:
    reference_path = ref_candidates[0]
else:
    raise FileNotFoundError("Reference raster not found in 04_Filtered_dB/")

# Input mask TIFF
mask_path = "Abhay Best.tif"

# Output
output_path = "crop_mask_aligned.tif"

if Path(mask_path).exists():
    with rasterio.open(reference_path) as ref, rasterio.open(mask_path) as src:
        # Output array matching reference
        mask = np.zeros((ref.height, ref.width), dtype=np.uint8)

        # Reproject/resample mask to reference grid
        reproject(
            source=rasterio.band(src, 1),
            destination=mask,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            resampling=Resampling.nearest  # Preserve binary values
        )

        # Ensure binary values (0/1)
        mask = (mask > 0).astype(np.uint8)

        profile = ref.profile.copy()
        profile.update(
            dtype=rasterio.uint8,
            count=1,
            compress="deflate",
            nodata=0
        )

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(mask, 1)

    print(f"Saved aligned binary mask to: {output_path}")
else:
    print(f"Mask file {mask_path} not found.")

## Step 2: Variance Analysis
A streaming pass (Welford's algorithm) over all features calculates their variance. Features with extremely low variance are flagged for removal to prevent instability in scaling and PCA steps.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
ROOT = Path(".")
OUT_DIR = ROOT / "15_NearZeroVariance"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FOLDERS = [
    "04_Filtered_dB",
    "05_Temporal_Stats",
    "06_Fast_Texture",
    "07_Terrain_Features",
    "08_Temporal_Dynamics",
    "09_Fast_Texture",
    "10_Fast_Texture",
]

EXCLUDE_FILENAMES = {
    "Temporal_ValidCount.tif",
}

BLOCK_SIZE = 1024
VAR_REMOVE_THRESH = 1e-6
STD_REVIEW_THRESH = 1e-3
RANGE_REVIEW_THRESH = 1e-3

def collect_feature_files():
    items = []
    for folder in FEATURE_FOLDERS:
        folder_path = ROOT / folder
        if not folder_path.exists():
            continue
        for tif in sorted(folder_path.glob("*.tif")):
            if tif.name in EXCLUDE_FILENAMES:
                continue
            feature_id = f"{folder}/{tif.stem}"
            items.append((feature_id, tif))
    if not items:
        raise RuntimeError("No raster features found.")
    return items

def open_sources(items):
    return [(feature_id, rasterio.open(path)) for feature_id, path in items]

def close_sources(sources):
    for _, src in sources:
        src.close()

def check_alignment(sources):
    ref_id, ref = sources[0]
    for feature_id, src in sources[1:]:
        if (
            src.crs != ref.crs
            or src.width != ref.width
            or src.height != ref.height
            or src.transform != ref.transform
        ):
            raise RuntimeError(
                f"Alignment mismatch:\nReference: {ref_id}\nProblem:   {feature_id}"
            )

def get_nodata(src):
    return src.nodata if src.nodata is not None else -9999.0

def iter_windows(width, height, block_size):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            w = min(block_size, width - col_off)
            h = min(block_size, height - row_off)
            yield Window(col_off, row_off, w, h)

def finite_valid(arr, nodata):
    return np.isfinite(arr) & (arr != nodata)

# ============================================================
# EXACT STREAMING VARIANCE (WELFORD)
# ============================================================
class RunningStats:
    def __init__(self):
        self.count = 0
        self.mean = 0.0
        self.M2 = 0.0
        self.min = np.inf
        self.max = -np.inf

    def update_array(self, x):
        x = np.asarray(x, dtype=np.float64)
        if x.size == 0:
            return

        x_min = float(np.min(x))
        x_max = float(np.max(x))
        if x_min < self.min:
            self.min = x_min
        if x_max > self.max:
            self.max = x_max

        n = x.size
        batch_mean = float(np.mean(x))
        batch_M2 = float(np.sum((x - batch_mean) ** 2))

        if self.count == 0:
            self.count = n
            self.mean = batch_mean
            self.M2 = batch_M2
            return

        delta = batch_mean - self.mean
        total = self.count + n
        self.mean = self.mean + delta * (n / total)
        self.M2 = self.M2 + batch_M2 + (delta * delta) * self.count * n / total
        self.count = total

    @property
    def variance(self):
        return self.M2 / self.count if self.count > 0 else np.nan

    @property
    def std(self):
        v = self.variance
        return math.sqrt(v) if np.isfinite(v) else np.nan

# ============================================================
# MAIN ANALYSIS
# ============================================================
def main():
    items = collect_feature_files()
    print(f"Found {len(items)} rasters")

    sources = open_sources(items)
    try:
        check_alignment(sources)
        print("Alignment check passed.")
        print("Scanning rasters for exact variance...")

        rows = []

        for feature_id, src in sources:
            nodata = get_nodata(src)
            stats = RunningStats()

            for window in iter_windows(src.width, src.height, BLOCK_SIZE):
                arr = src.read(1, window=window).astype(np.float32, copy=False)
                valid = finite_valid(arr, nodata)
                if not np.any(valid):
                    continue

                vals = arr[valid]
                stats.update_array(vals)

                del arr, valid, vals

            var = stats.variance
            std = stats.std
            data_range = stats.max - stats.min if np.isfinite(stats.min) and np.isfinite(stats.max) else np.nan
            cv = abs(std / stats.mean) if np.isfinite(std) and np.isfinite(stats.mean) and abs(stats.mean) > 1e-12 else np.nan

            remove_flag = bool(
                (np.isfinite(var) and var < VAR_REMOVE_THRESH)
            )
            review_flag = bool(
                (np.isfinite(std) and std < STD_REVIEW_THRESH) or
                (np.isfinite(data_range) and data_range < RANGE_REVIEW_THRESH)
            )

            rows.append({
                "feature_id": feature_id,
                "valid_count": int(stats.count),
                "min": float(stats.min) if np.isfinite(stats.min) else np.nan,
                "max": float(stats.max) if np.isfinite(stats.max) else np.nan,
                "mean": float(stats.mean) if np.isfinite(stats.mean) else np.nan,
                "variance": float(var) if np.isfinite(var) else np.nan,
                "std": float(std) if np.isfinite(std) else np.nan,
                "range": float(data_range) if np.isfinite(data_range) else np.nan,
                "cv": float(cv) if np.isfinite(cv) else np.nan,
                "near_zero_remove": remove_flag,
                "review": review_flag,
            })

        df = pd.DataFrame(rows).sort_values("variance", ascending=True)
        df.to_csv(OUT_DIR / "variance_scores.csv", index=False)

        near_zero = df[df["near_zero_remove"] | df["review"]].copy()
        near_zero.to_csv(OUT_DIR / "near_zero_variance.csv", index=False)

        # Plot
        plt.figure(figsize=(12, max(6, 0.28 * len(df))))
        plt.barh(df["feature_id"], df["variance"])
        plt.axvline(VAR_REMOVE_THRESH, linestyle="--")
        plt.axvline(STD_REVIEW_THRESH ** 2, linestyle="--")
        plt.gca().invert_yaxis()
        plt.xlabel("Variance")
        plt.title("Exact Variance by Feature")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / "variance_barplot.png", dpi=180)
        plt.close()

        # Markdown report
        lines = []
        lines.append("# Near-Zero Variance Report")
        lines.append("")
        lines.append(f"- Variance remove threshold: {VAR_REMOVE_THRESH}")
        lines.append(f"- Std review threshold: {STD_REVIEW_THRESH}")
        lines.append(f"- Range review threshold: {RANGE_REVIEW_THRESH}")
        lines.append("")
        lines.append("## Features flagged for removal/review")
        if len(near_zero) == 0:
            lines.append("- None.")
        else:
            for _, r in near_zero.iterrows():
                lines.append(
                    f"- {r['feature_id']}: variance={r['variance']:.6g}, "
                    f"std={r['std']:.6g}, range={r['range']:.6g}, "
                    f"remove={bool(r['near_zero_remove'])}, review={bool(r['review'])}"
                )

        lines.append("")
        lines.append("## Lowest-variance features")
        for _, r in df.head(15).iterrows():
            lines.append(
                f"- {r['feature_id']}: variance={r['variance']:.6g}, std={r['std']:.6g}, range={r['range']:.6g}"
            )

        (OUT_DIR / "variance_report.md").write_text("\n".join(lines), encoding="utf-8")

        meta = {
            "block_size": BLOCK_SIZE,
            "variance_remove_threshold": VAR_REMOVE_THRESH,
            "std_review_threshold": STD_REVIEW_THRESH,
            "range_review_threshold": RANGE_REVIEW_THRESH,
            "n_features": int(len(df)),
        }
        (OUT_DIR / "variance_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

        print("\nTop low-variance features:")
        print(df.head(15).to_string(index=False))

        print(f"\nSaved to: {OUT_DIR}")

    finally:
        close_sources(sources)

if __name__ == "__main__":
    main()

## Step 3: Grouped PCA Feature Extraction
We extract principal components grouping features logically (Temporal, Texture, Terrain). This version incorporates an enhancement where temporal ratios are computed dynamically before PCA, helping capture the crop phenological differences over time. The processing is strictly confined to masked agricultural pixels.

In [ ]:
from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
import joblib
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
ROOT = Path(".")
OUT_DIR = ROOT / "11_PCA"
MODEL_DIR = OUT_DIR / "models"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Agricultural mask from the previous cell
MASK_PATH = Path("crop_mask_aligned.tif")
USE_AG_MASK = MASK_PATH.exists()
if USE_AG_MASK:
    print(f"✅ Agricultural mask found: {MASK_PATH}")
else:
    print("⚠ No agricultural mask found — processing ALL pixels (not recommended)")

FEATURE_FOLDERS = [
    "04_Filtered_dB",
    "05_Temporal_Stats",
    "06_Fast_Texture",
    "07_Terrain_Features",
    "08_Temporal_Dynamics",
    "09_Fast_Texture",
    "10_Fast_Texture",
]

EXCLUDE_FILENAMES = {
    "Temporal_ValidCount.tif",
}

# If you later decide to remove Temporal_Std, add it here:
MANUAL_DROP_SUBSTRINGS = {
    # "Temporal_Std",
}

USE_ASPECT_SIN_COS = True

BLOCK_SIZE = 2048
PCA_BATCH_ROWS = 200_000  # only used if a block is huge

N_COMP_TEMPORAL = 4
N_COMP_TEXTURE = 4
N_COMP_TERRAIN = 2
N_COMPONENTS = N_COMP_TEMPORAL + N_COMP_TEXTURE + N_COMP_TERRAIN

DEFAULT_NODATA = -9999.0
COMPRESS = "deflate"
ZLEVEL = 9

# ============================================================
# HELPERS
# ============================================================
def collect_feature_files():
    items = []
    for folder in FEATURE_FOLDERS:
        folder_path = ROOT / folder
        if not folder_path.exists():
            continue

        for tif in sorted(folder_path.glob("*.tif")):
            if tif.name in EXCLUDE_FILENAMES:
                continue
            if any(s in tif.name for s in MANUAL_DROP_SUBSTRINGS):
                continue
            feature_id = f"{folder}/{tif.stem}"
            items.append((feature_id, tif))

    if not items:
        raise RuntimeError("No feature rasters found.")

    return items

def open_sources(items):
    return [(feature_id, rasterio.open(path)) for feature_id, path in items]

def close_sources(sources):
    for _, src in sources:
        src.close()

def check_alignment(sources, mask_src=None):
    ref_id, ref = sources[0]
    for feature_id, src in sources[1:]:
        if (
            src.crs != ref.crs
            or src.width != ref.width
            or src.height != ref.height
            or src.transform != ref.transform
        ):
            raise RuntimeError(
                f"Alignment mismatch:\nReference: {ref_id}\nProblem:   {feature_id}"
            )
    if mask_src is not None:
        if (
            mask_src.crs != ref.crs
            or mask_src.width != ref.width
            or mask_src.height != ref.height
            or mask_src.transform != ref.transform
        ):
            raise RuntimeError(
                f"Alignment mismatch:\nReference: {ref_id}\nProblem:   Agricultural Mask"
            )

def get_nodata(src):
    return src.nodata if src.nodata is not None else DEFAULT_NODATA

def iter_windows(width, height, block_size):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            w = min(block_size, width - col_off)
            h = min(block_size, height - row_off)
            yield Window(col_off, row_off, w, h)

def feature_name(fid):
    return fid.split("/", 1)[1]

def is_aspect(fid):
    return "Aspect" in fid

def valid_mask(stack, nodatas, ag_mask_block=None):
    mask = np.isfinite(stack)
    for i in range(stack.shape[0]):
        mask[i] &= (stack[i] != nodatas[i])
    joint = np.all(mask, axis=0)
    if ag_mask_block is not None:
        joint &= (ag_mask_block == 1)
    return joint

def build_feature_vector(stack, nodatas, feature_ids, ag_mask_block=None):
    joint = valid_mask(stack, nodatas, ag_mask_block)
    if not np.any(joint):
        return None

    flat = stack.reshape(stack.shape[0], -1).T
    rows = flat[joint.ravel()].astype(np.float32, copy=False)

    if not USE_ASPECT_SIN_COS:
        return rows

    cols = []
    for i, fid in enumerate(feature_ids):
        if is_aspect(fid):
            ang = np.deg2rad(rows[:, i].astype(np.float32))
            cols.append(np.sin(ang)[:, None])
            cols.append(np.cos(ang)[:, None])
        else:
            cols.append(rows[:, i:i+1])

    base_features = np.hstack(cols).astype(np.float32, copy=False)

    temporal_cols = [i for i, fid in enumerate(feature_ids) if "04_Filtered_dB" in fid]
    if len(temporal_cols) > 1:
        ratio_cols = []
        for i in range(len(temporal_cols)):
            for j in range(i + 1, len(temporal_cols)):
                col_i = base_features[:, temporal_cols[i]]
                col_j = base_features[:, temporal_cols[j]]
                ratio = (col_j + 1e-6) / (col_i + 1e-6)
                ratio_cols.append(ratio[:, None])
        if ratio_cols:
            return np.hstack([base_features] + ratio_cols).astype(np.float32, copy=False)
    return base_features

def build_output_feature_names(feature_ids):
    names = []
    for fid in feature_ids:
        if USE_ASPECT_SIN_COS and is_aspect(fid):
            base = feature_name(fid)
            names.append(f"{base}_sin")
            names.append(f"{base}_cos")
        else:
            names.append(feature_name(fid))
    temporal_cols = [fid for fid in feature_ids if "04_Filtered_dB" in fid]
    for i in range(len(temporal_cols)):
        for j in range(i + 1, len(temporal_cols)):
            names.append(f"Ratio_{feature_name(temporal_cols[j])}_over_{feature_name(temporal_cols[i])}")
    return names

def get_group_indices(feature_ids):
    temporal_indices = []
    texture_indices = []
    terrain_indices = []

    curr_idx = 0
    for fid in feature_ids:
        folder = fid.split("/")[0]
        
        num_feats = 2 if (USE_ASPECT_SIN_COS and is_aspect(fid)) else 1
            
        for _ in range(num_feats):
            if folder in ["04_Filtered_dB", "08_Temporal_Dynamics", "05_Temporal_Stats"]:
                temporal_indices.append(curr_idx)
            elif folder in ["06_Fast_Texture", "09_Fast_Texture", "10_Fast_Texture"]:
                texture_indices.append(curr_idx)
            elif folder in ["07_Terrain_Features"]:
                terrain_indices.append(curr_idx)
            curr_idx += 1
            
    temporal_cols = [fid for fid in feature_ids if "04_Filtered_dB" in fid]
    num_ratios = len(temporal_cols) * (len(temporal_cols) - 1) // 2
    for _ in range(num_ratios):
        temporal_indices.append(curr_idx)
        curr_idx += 1
    return temporal_indices, texture_indices, terrain_indices

def fit_scaler(sources, feature_ids, mask_src=None):
    ref = sources[0][1]
    scaler = StandardScaler(with_mean=True, with_std=True)

    total = 0
    for window in iter_windows(ref.width, ref.height, BLOCK_SIZE):
        arrays = []
        nodatas = []
        for _, src in sources:
            arrays.append(src.read(1, window=window))
            nodatas.append(get_nodata(src))

        ag_block = None
        if mask_src is not None:
            ag_block = mask_src.read(1, window=window)

        stack = np.stack(arrays, axis=0)
        nodatas = np.asarray(nodatas, dtype=np.float32)

        X = build_feature_vector(stack, nodatas, feature_ids, ag_block)
        if X is not None and X.shape[0] > 0:
            scaler.partial_fit(X)
            total += X.shape[0]

        del arrays, nodatas, stack, X, ag_block
        gc.collect()

    return scaler, total


def fit_grouped_ipca(sources, feature_ids, scaler, temporal_indices, texture_indices, terrain_indices, mask_src=None):
    ref = sources[0][1]
    
    pca_temp = IncrementalPCA(n_components=N_COMP_TEMPORAL)
    pca_text = IncrementalPCA(n_components=N_COMP_TEXTURE)
    pca_terr = IncrementalPCA(n_components=N_COMP_TERRAIN)

    total = 0
    for window in iter_windows(ref.width, ref.height, BLOCK_SIZE):
        arrays = []
        nodatas = []
        for _, src in sources:
            arrays.append(src.read(1, window=window))
            nodatas.append(get_nodata(src))

        ag_block = None
        if mask_src is not None:
            ag_block = mask_src.read(1, window=window)

        stack = np.stack(arrays, axis=0)
        nodatas = np.asarray(nodatas, dtype=np.float32)

        X = build_feature_vector(stack, nodatas, feature_ids, ag_block)
        if X is None or X.shape[0] == 0:
            del arrays, nodatas, stack, X, ag_block
            gc.collect()
            continue

        Xs = scaler.transform(X).astype(np.float32, copy=False)

        if Xs.shape[0] > 0:
            for start in range(0, Xs.shape[0], PCA_BATCH_ROWS):
                batch = Xs[start:start + PCA_BATCH_ROWS]
                
                b_temp = batch[:, temporal_indices]
                if b_temp.shape[0] >= N_COMP_TEMPORAL:
                    pca_temp.partial_fit(b_temp)
                    
                b_text = batch[:, texture_indices]
                if b_text.shape[0] >= N_COMP_TEXTURE:
                    pca_text.partial_fit(b_text)
                    
                b_terr = batch[:, terrain_indices]
                if b_terr.shape[0] >= N_COMP_TERRAIN:
                    pca_terr.partial_fit(b_terr)
                    
                if start == 0: # Only count total rows once per window
                    total += batch.shape[0]

        del arrays, nodatas, stack, X, Xs, ag_block
        gc.collect()

    return pca_temp, pca_text, pca_terr, total


def write_grouped_pca_raster(sources, feature_ids, scaler, pca_temp, pca_text, pca_terr, 
                           temporal_indices, texture_indices, terrain_indices, out_feature_names, mask_src=None):
    ref = sources[0][1]
    out_meta = ref.meta.copy()
    out_meta.update(
        dtype="float32",
        count=N_COMPONENTS,
        nodata=DEFAULT_NODATA,
        compress=COMPRESS,
        zlevel=ZLEVEL,
        predictor=3,
        tiled=True,
        blockxsize=512,
        blockysize=512,
        BIGTIFF="IF_SAFER",
    )

    out_path = OUT_DIR / "PCA_Components.tif"

    with open(OUT_DIR / "pca_used_features.json", "w") as f:
        json.dump(out_feature_names, f, indent=2)

    with rasterio.open(out_path, "w", **out_meta) as dst:
        for window in iter_windows(ref.width, ref.height, BLOCK_SIZE):
            arrays = []
            nodatas = []
            for _, src in sources:
                arrays.append(src.read(1, window=window))
                nodatas.append(get_nodata(src))

            ag_block = None
            if mask_src is not None:
                ag_block = mask_src.read(1, window=window)

            stack = np.stack(arrays, axis=0)
            nodatas = np.asarray(nodatas, dtype=np.float32)

            X = build_feature_vector(stack, nodatas, feature_ids, ag_block)

            out_block = np.full(
                (N_COMPONENTS, window.height, window.width),
                DEFAULT_NODATA,
                dtype=np.float32,
            )

            if X is not None and X.shape[0] > 0:
                Xs = scaler.transform(X).astype(np.float32, copy=False)
                
                P_temp = pca_temp.transform(Xs[:, temporal_indices]).astype(np.float32, copy=False)
                P_text = pca_text.transform(Xs[:, texture_indices]).astype(np.float32, copy=False)
                P_terr = pca_terr.transform(Xs[:, terrain_indices]).astype(np.float32, copy=False)
                
                P = np.hstack([P_temp, P_text, P_terr])

                joint_valid = valid_mask(stack, nodatas, ag_block).ravel()
                flat_out = np.full(
                    (window.width * window.height, N_COMPONENTS),
                    DEFAULT_NODATA,
                    dtype=np.float32,
                )
                flat_out[joint_valid, :] = P
                out_block = flat_out.reshape(window.height, window.width, N_COMPONENTS).transpose(2, 0, 1)

                del Xs, P_temp, P_text, P_terr, P, flat_out

            for i in range(N_COMPONENTS):
                dst.write(out_block[i], i + 1, window=window)

            del arrays, nodatas, stack, X, out_block, ag_block
            gc.collect()

    return out_path


# ============================================================
# MAIN
# ============================================================
def main():
    items = collect_feature_files()
    print(f"Found {len(items)} rasters")

    sources = open_sources(items)
    mask_src = rasterio.open(MASK_PATH) if USE_AG_MASK else None

    try:
        check_alignment(sources, mask_src)
        print("Alignment check passed.")

        feature_ids = [fid for fid, _ in sources]
        out_feature_names = build_output_feature_names(feature_ids)
        temporal_idx, texture_idx, terrain_idx = get_group_indices(feature_ids)
        
        print(f"Feature groups:")
        print(f"  - Temporal: {len(temporal_idx)} features")
        print(f"  - Texture:  {len(texture_idx)} features")
        print(f"  - Terrain:  {len(terrain_idx)} features")

        print("\nFitting StandardScaler (cropland pixels only)..." if USE_AG_MASK else "\nFitting StandardScaler...")
        scaler, fit_pixels = fit_scaler(sources, feature_ids, mask_src)
        joblib.dump(scaler, MODEL_DIR / "standard_scaler.joblib")
        print(f"  → Scaler fit on {fit_pixels:,} pixels")

        print("\nFitting Grouped IncrementalPCA (cropland pixels only)..." if USE_AG_MASK else "\nFitting Grouped IncrementalPCA...")
        pca_temp, pca_text, pca_terr, pca_pixels = fit_grouped_ipca(
            sources, feature_ids, scaler, temporal_idx, texture_idx, terrain_idx, mask_src
        )
        joblib.dump(pca_temp, MODEL_DIR / "pca_temporal.joblib")
        joblib.dump(pca_text, MODEL_DIR / "pca_texture.joblib")
        joblib.dump(pca_terr, MODEL_DIR / "pca_terrain.joblib")
        print(f"  → Grouped PCA fit on {pca_pixels:,} pixels")

        print("\nSaving PCA outputs...")
        out_path = write_grouped_pca_raster(
            sources, feature_ids, scaler, pca_temp, pca_text, pca_terr,
            temporal_idx, texture_idx, terrain_idx, out_feature_names, mask_src
        )

        meta = {
            "n_input_rasters": len(feature_ids),
            "n_output_components": N_COMPONENTS,
            "components_breakdown": {
                "temporal": N_COMP_TEMPORAL,
                "texture": N_COMP_TEXTURE,
                "terrain": N_COMP_TERRAIN
            },
            "block_size": BLOCK_SIZE,
            "use_aspect_sin_cos": USE_ASPECT_SIN_COS,
            "feature_names_used": out_feature_names,
            "scaler_fit_pixels": int(fit_pixels),
            "pca_fit_pixels": int(pca_pixels),
            "agricultural_mask_applied": USE_AG_MASK,
        }
        (OUT_DIR / "pca_meta.json").write_text(json.dumps(meta, indent=2))

        print(f"\n✅ Done. Grouped PCA raster saved to: {out_path}")

    finally:
        close_sources(sources)
        if mask_src is not None:
            mask_src.close()


if __name__ == "__main__":
    main()

## Step 4: Primary GMM Clustering
We use a Gaussian Mixture Model (GMM) with `full` covariance matrices. To pick the optimal number of clusters (K), we calculate BIC, AIC, and Silhouette scores. The resulting cluster map is passed through a 3x3 majority filter to reduce spatial speckle.

In [ ]:
from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import joblib

# ============================================================
# CONFIG
# ============================================================
ROOT = Path(".")
PCA_PATH = ROOT / "11_PCA" / "PCA_Components.tif"
OUT_DIR = ROOT / "16_GMM"
MODEL_DIR = OUT_DIR / "models"
PLOT_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

K_MIN = 2
K_MAX = 10

COVARIANCE_TYPE = "full"
RANDOM_STATE = 42
N_INIT = 5
MAX_ITER = 200
REG_COVAR = 1e-6

MAX_SAMPLE_PIXELS = 300_000
BLOCK_SIZE = 2048
SEED = 42

NODATA_OUT = 0
CLUSTER_OFFSET = 1

rng = np.random.default_rng(SEED)

# ============================================================
# HELPERS
# ============================================================
def iter_windows(width, height, block_size):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            w = min(block_size, width - col_off)
            h = min(block_size, height - row_off)
            yield Window(col_off, row_off, w, h)


def valid_mask_from_stack(stack, nodata):
    mask = np.isfinite(stack) & (stack != nodata)
    return np.all(mask, axis=0)


def reservoir_init(max_samples, n_features):
    reservoir = np.empty((max_samples, n_features), dtype=np.float32)
    filled = 0
    seen = 0
    return reservoir, filled, seen


def reservoir_update(reservoir, filled, seen, rows, max_samples, rng):
    """
    Exact reservoir sampling (Algorithm R).
    Every valid row seen so far has equal probability of being in the reservoir.
    """
    if rows is None or rows.shape[0] == 0:
        return reservoir, filled, seen

    for row in rows:
        seen += 1

        if filled < max_samples:
            reservoir[filled] = row
            filled += 1
        else:
            j = rng.integers(0, seen)
            if j < max_samples:
                reservoir[j] = row

    return reservoir, filled, seen


def load_sample_from_pca(pca_path):
    with rasterio.open(pca_path) as src:
        nodata = src.nodata if src.nodata is not None else -9999.0
        n_features = src.count

        reservoir, filled, seen = reservoir_init(MAX_SAMPLE_PIXELS, n_features)

        for window in iter_windows(src.width, src.height, BLOCK_SIZE):
            stack = src.read(window=window).astype(np.float32, copy=False)  # (bands, h, w)
            joint_valid = valid_mask_from_stack(stack, nodata)

            if not np.any(joint_valid):
                del stack, joint_valid
                continue

            flat = stack.reshape(stack.shape[0], -1).T
            rows = flat[joint_valid.ravel()].astype(np.float32, copy=False)

            reservoir, filled, seen = reservoir_update(
                reservoir, filled, seen, rows, MAX_SAMPLE_PIXELS, rng
            )

            del stack, joint_valid, flat, rows
            gc.collect()

        if filled == 0:
            raise RuntimeError("No valid PCA pixels found for GMM fitting.")

        return reservoir[:filled].copy(), seen


def fit_best_gmm(X):
    results = []
    best_bic = np.inf
    best_model = None
    best_k = None

    for k in range(K_MIN, K_MAX + 1):
        print(f"Fitting GMM with K={k}...")

        gmm = GaussianMixture(
            n_components=k,
            covariance_type=COVARIANCE_TYPE,
            random_state=RANDOM_STATE,
            n_init=N_INIT,
            max_iter=MAX_ITER,
            reg_covar=REG_COVAR,
            init_params="kmeans",
        )
        gmm.fit(X)

        bic = gmm.bic(X)
        aic = gmm.aic(X)

        print("  Calculating silhouette score...")
        if X.shape[0] > 10000:
            idx = rng.choice(X.shape[0], 10000, replace=False)
            sil = silhouette_score(X[idx], gmm.predict(X[idx]))
        else:
            sil = silhouette_score(X, gmm.predict(X))

        results.append(
            {
                "k": k,
                "bic": bic,
                "aic": aic,
                "silhouette": sil,
                "converged": bool(gmm.converged_),
                "n_iter": int(gmm.n_iter_),
            }
        )

        if bic < best_bic:
            best_bic = bic
            best_model = gmm
            best_k = k

        print(f"  BIC={bic:.3f}  AIC={aic:.3f}  converged={gmm.converged_}")

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUT_DIR / "gmm_bic_aic_scores.csv", index=False)

    plt.figure(figsize=(10, 5))
    plt.plot(results_df["k"], results_df["bic"], marker="o", label="BIC")
    plt.plot(results_df["k"], results_df["aic"], marker="o", label="AIC")
    plt.xlabel("Number of Clusters (K)")
    plt.ylabel("Score")
    plt.title("GMM Model Selection")
    plt.legend(loc="upper left")
    
    ax2 = plt.gca().twinx()
    ax2.plot(results_df["k"], results_df["silhouette"], marker="s", color="red", label="Silhouette")
    ax2.set_ylabel("Silhouette Score", color="red")
    ax2.tick_params(axis='y', labelcolor="red")
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "gmm_bic_aic.png", dpi=180)
    plt.close()

    return best_model, best_k, results_df


from scipy.ndimage import generic_filter

def majority_filter(arr, size=3):
    def _majority(values):
        values = values[values > 0]
        if len(values) == 0:
            return 0
        unique, counts = np.unique(values, return_counts=True)
        return unique[np.argmax(counts)]
    return generic_filter(arr.astype(float), _majority, size=size)

def predict_full_raster(pca_path, model, out_path):
    with rasterio.open(pca_path) as src:
        nodata = src.nodata if src.nodata is not None else -9999.0

        out_meta = src.meta.copy()
        out_meta.update(
            count=1,
            dtype="uint16",
            nodata=NODATA_OUT,
            compress="deflate",
            zlevel=9,
            predictor=2,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(out_path, "w", **out_meta) as dst:
            for window in iter_windows(src.width, src.height, BLOCK_SIZE):
                stack = src.read(window=window).astype(np.float32, copy=False)
                valid = valid_mask_from_stack(stack, nodata)

                out_block = np.full((window.height, window.width), NODATA_OUT, dtype=np.uint16)

                if np.any(valid):
                    flat = stack.reshape(stack.shape[0], -1).T
                    X = flat[valid.ravel()].astype(np.float32, copy=False)

                    labels = model.predict(X).astype(np.uint16) + CLUSTER_OFFSET
                    out_block.ravel()[valid.ravel()] = labels

                    del flat, X, labels

                if np.any(valid):
                    out_block = majority_filter(out_block, size=3).astype(np.uint16)
                dst.write(out_block, 1, window=window)

                del stack, valid, out_block
                gc.collect()


def save_report(best_k, sample_shape, total_valid_seen):
    report = []
    report.append("# GMM Clustering Report")
    report.append("")
    report.append(f"- PCA input: {PCA_PATH}")
    report.append(f"- Sample shape: {sample_shape[0]:,} × {sample_shape[1]}")
    report.append(f"- Total valid pixels seen: {total_valid_seen:,}")
    report.append(f"- Best K by BIC: {best_k}")
    report.append(f"- Covariance type: {COVARIANCE_TYPE}")
    report.append(f"- N_INIT: {N_INIT}")
    report.append(f"- MAX_ITER: {MAX_ITER}")
    report.append(f"- REG_COVAR: {REG_COVAR}")
    report.append("")
    report.append("Cluster labels in output raster start at 1. Nodata is 0.")

    (OUT_DIR / "gmm_report.md").write_text("\n".join(report), encoding="utf-8")


# ============================================================
# MAIN
# ============================================================
def main():
    if not PCA_PATH.exists():
        raise FileNotFoundError(f"Missing PCA raster: {PCA_PATH}")

    print("Loading PCA sample...")
    X, total_valid_seen = load_sample_from_pca(PCA_PATH)
    print(f"Sample collected: {X.shape}")

    print("Selecting best GMM...")
    best_model, best_k, scores_df = fit_best_gmm(X)

    print(f"Best K = {best_k}")
    joblib.dump(best_model, MODEL_DIR / "gmm_best.joblib")

    print("Predicting full raster...")
    out_path = OUT_DIR / "GMM_Cluster_Map.tif"
    predict_full_raster(PCA_PATH, best_model, out_path)

    save_report(best_k, X.shape, total_valid_seen)

    meta = {
        "pca_path": str(PCA_PATH),
        "output_raster": str(out_path),
        "best_k": int(best_k),
        "sample_shape": [int(X.shape[0]), int(X.shape[1])],
        "total_valid_pixels_seen": int(total_valid_seen),
        "covariance_type": COVARIANCE_TYPE,
        "k_range": [K_MIN, K_MAX],
        "sampling": "true_reservoir_sampling_algorithm_R",
    }
    (OUT_DIR / "gmm_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print(f"Done. Saved: {out_path}")


if __name__ == "__main__":
    main()


## Step 5: Hierarchical Reclustering
Certain clusters often represent mixtures of different crop types that are difficult to separate globally. We perform a secondary hierarchical clustering strictly on the pixels of a specified `target_cluster`, effectively splitting it into smaller subclusters while retaining the original clustering ID space.

In [ ]:
from __future__ import annotations

import argparse
import json
import logging
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import joblib
import matplotlib

# Kaggle notebook cells run headless (no display). "Agg" renders figures to
# file without needing a GUI backend. Must be set before pyplot is imported.
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIG
# ============================================================
# On Kaggle, /kaggle/input is READ-ONLY (your uploaded dataset lives there)
# and /kaggle/working is the ONLY writable location (this is also what gets
# saved when you "Save Version" / commit the notebook). Everywhere else
# (e.g. a local machine), fall back to the current directory for both.
def _is_kaggle() -> bool:
    return Path("/kaggle").exists()


ROOT = Path(".")
DEFAULT_OUT_ROOT = Path("/kaggle/working") if _is_kaggle() else Path(".")

TARGET_CLUSTER_DEFAULT = 3
K_MIN_DEFAULT = 2
K_MAX_DEFAULT = 8
RANDOM_STATE_DEFAULT = 42
COVARIANCE_TYPE_DEFAULT = "diag"
REG_COVAR_DEFAULT = 1e-6
N_INIT_DEFAULT = 5
MAX_ITER_DEFAULT = 300
SEARCH_SAMPLE_SIZE_DEFAULT = 200_000
VALIDATION_FRACTION_DEFAULT = 0.2
SILHOUETTE_SAMPLE_SIZE_DEFAULT = 5_000
PRED_BATCH_SIZE_DEFAULT = 250_000
MIN_SUBCLUSTER_FRACTION_DEFAULT = 0.01  # flag sub-clusters smaller than 1% of the target cluster
NODATA_OUT_DEFAULT = -9999


# ============================================================
# LOGGING
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s: %(message)s",
)
logger = logging.getLogger("hierarchical_gmm")


@dataclass
class WindowStat:
    window: Window
    target_count: int


# ============================================================
# HELPERS
# ============================================================
def parse_band_subset(spec: Optional[str], n_bands: int) -> np.ndarray:
    """Parse a 1-indexed, comma-separated band list (e.g. '1,2,3,4,5'). None = all bands."""
    if not spec:
        return np.arange(n_bands)
    idx = [int(x.strip()) - 1 for x in spec.split(",") if x.strip()]
    for i in idx:
        if i < 0 or i >= n_bands:
            raise ValueError(f"Band index out of range: {i + 1} (raster has {n_bands} bands)")
    return np.array(idx, dtype=int)


def ensure_alignment(pca_ds: rasterio.io.DatasetReader, cluster_ds: rasterio.io.DatasetReader) -> None:
    """Hard fail if rasters are not perfectly aligned."""
    checks = {
        "width": pca_ds.width == cluster_ds.width,
        "height": pca_ds.height == cluster_ds.height,
        "crs": pca_ds.crs == cluster_ds.crs,
        "transform": pca_ds.transform == cluster_ds.transform,
        "count_cluster_single_band": cluster_ds.count == 1,
        "pca_multi_band": pca_ds.count >= 2,
    }
    bad = [name for name, ok in checks.items() if not ok]
    if bad:
        raise ValueError("Raster alignment check failed for: " + ", ".join(bad))


def get_nodata_value(ds: rasterio.io.DatasetReader, fallback: float = np.nan) -> float:
    return fallback if ds.nodata is None else ds.nodata


def read_window_arrays(
    pca_ds: rasterio.io.DatasetReader,
    cluster_ds: rasterio.io.DatasetReader,
    window: Window,
    band_idx: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Read PCA as (H, W, B) float32 (only selected bands) and cluster as (H, W) int32."""
    pca = pca_ds.read(window=window).astype(np.float32, copy=False)  # (B_all, H, W)
    pca = pca[band_idx, :, :]
    pca = np.moveaxis(pca, 0, -1)  # (H, W, B)
    cluster = cluster_ds.read(1, window=window).astype(np.int32, copy=False)
    return pca, cluster


def make_valid_mask(
    pca: np.ndarray,
    cluster: np.ndarray,
    target_cluster: int,
    pca_nodata: float,
    cluster_nodata: float,
) -> np.ndarray:
    mask = cluster == target_cluster
    if not np.isnan(cluster_nodata):
        mask &= cluster != cluster_nodata
    if not np.isnan(pca_nodata):
        # Exact equality is fine here: nodata is written verbatim by the raster
        # producer, not derived from floating-point arithmetic.
        mask &= np.all(pca != pca_nodata, axis=-1)
    mask &= np.all(np.isfinite(pca), axis=-1)
    return mask


def collect_window_stats(
    pca_ds: rasterio.io.DatasetReader,
    cluster_ds: rasterio.io.DatasetReader,
    target_cluster: int,
    band_idx: np.ndarray,
) -> List[WindowStat]:
    stats: List[WindowStat] = []
    pca_nodata = get_nodata_value(pca_ds)
    cluster_nodata = get_nodata_value(cluster_ds)

    for _, window in cluster_ds.block_windows(1):
        pca, cluster = read_window_arrays(pca_ds, cluster_ds, window, band_idx)
        mask = make_valid_mask(pca, cluster, target_cluster, pca_nodata, cluster_nodata)
        count = int(mask.sum())
        if count > 0:
            stats.append(WindowStat(window=window, target_count=count))

    return stats


def allocate_sample_quotas(stats: List[WindowStat], sample_size: int) -> List[int]:
    total = sum(s.target_count for s in stats)
    if total == 0:
        return [0] * len(stats)

    if total <= sample_size:
        return [s.target_count for s in stats]

    raw = [sample_size * s.target_count / total for s in stats]
    quotas = [int(math.floor(x)) for x in raw]
    remainder = sample_size - sum(quotas)

    fractional = sorted(
        enumerate([x - math.floor(x) for x in raw]),
        key=lambda t: t[1],
        reverse=True,
    )
    for idx, _ in fractional[:remainder]:
        quotas[idx] += 1

    return quotas


def build_sample(
    pca_ds: rasterio.io.DatasetReader,
    cluster_ds: rasterio.io.DatasetReader,
    target_cluster: int,
    sample_size: int,
    random_state: int,
    band_idx: np.ndarray,
) -> np.ndarray:
    """Stratified sample over block windows; suitable for model selection and fitting."""
    rng = np.random.default_rng(random_state)
    pca_nodata = get_nodata_value(pca_ds)
    cluster_nodata = get_nodata_value(cluster_ds)

    stats = collect_window_stats(pca_ds, cluster_ds, target_cluster, band_idx)
    total_target = sum(s.target_count for s in stats)
    if total_target == 0:
        raise ValueError(f"Target cluster {target_cluster} not found in the cluster raster.")

    quotas = allocate_sample_quotas(stats, sample_size)
    logger.info("Target pixels in cluster %s: %s", target_cluster, total_target)
    logger.info("Sampling up to %s pixels from cluster %s", min(sample_size, total_target), target_cluster)

    samples = []
    for stat, quota in zip(stats, quotas):
        if quota <= 0:
            continue
        pca, cluster = read_window_arrays(pca_ds, cluster_ds, stat.window, band_idx)
        mask = make_valid_mask(pca, cluster, target_cluster, pca_nodata, cluster_nodata)
        feats = pca[mask]
        if feats.shape[0] == 0:
            continue
        if feats.shape[0] > quota:
            idx = rng.choice(feats.shape[0], size=quota, replace=False)
            feats = feats[idx]
        samples.append(feats)

    if not samples:
        raise ValueError("No valid pixels collected for the sample.")

    X = np.concatenate(samples, axis=0).astype(np.float32, copy=False)
    if X.shape[0] > sample_size:
        idx = rng.choice(X.shape[0], size=sample_size, replace=False)
        X = X[idx]

    logger.info("Final sample shape: %s", X.shape)
    return X


def fit_best_gmm(
    X: np.ndarray,
    k_min: int,
    k_max: int,
    covariance_type: str,
    reg_covar: float,
    n_init: int,
    max_iter: int,
    random_state: int,
    validation_fraction: float,
    silhouette_sample_size: int,
    out_dir: Path,
) -> Tuple[GaussianMixture, StandardScaler, pd.DataFrame, int]:
    n_samples = X.shape[0]
    if n_samples < 2:
        raise ValueError("Need at least 2 samples to fit GMM.")

    # --- Standardize features -------------------------------------------------
    # PCA components typically have very different variances (PC1 >> PC10).
    # GMM's covariance estimation and reg_covar are both scale-sensitive:
    # an absolute reg_covar can be negligible on high-variance bands and
    # dominate/regularize away real structure on low-variance bands.
    # Standardizing puts every band on equal footing before fitting.
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X).astype(np.float32, copy=False)

    # --- Train / validation split for an N-bias-robust selection signal -------
    X_train, X_val = train_test_split(
        Xs, test_size=validation_fraction, random_state=random_state
    )

    effective_k_max = min(k_max, X_train.shape[0] - 1)
    if effective_k_max < k_min:
        raise ValueError(f"Not enough samples ({n_samples}) for K range {k_min}..{k_max}.")

    rng = np.random.default_rng(random_state)
    sil_n = min(silhouette_sample_size, Xs.shape[0])
    sil_idx = rng.choice(Xs.shape[0], size=sil_n, replace=False)
    X_sil = Xs[sil_idx]

    records = []
    models = {}

    for k in range(k_min, effective_k_max + 1):
        logger.info("Fitting candidate GMM with K=%s", k)
        gmm = GaussianMixture(
            n_components=k,
            covariance_type=covariance_type,
            reg_covar=reg_covar,
            n_init=n_init,
            max_iter=max_iter,
            random_state=random_state,
            init_params="kmeans",
        )
        # Fit on the full standardized sample for the final model...
        gmm.fit(Xs)
        aic = gmm.aic(Xs)
        bic = gmm.bic(Xs)

        # ...but also fit on the train split alone to score held-out likelihood.
        # This guards against BIC's tendency to keep favoring larger K as N grows.
        gmm_train = GaussianMixture(
            n_components=k,
            covariance_type=covariance_type,
            reg_covar=reg_covar,
            n_init=n_init,
            max_iter=max_iter,
            random_state=random_state,
            init_params="kmeans",
        )
        gmm_train.fit(X_train)
        val_ll = float(gmm_train.score(X_val))  # mean log-likelihood per sample

        sil = np.nan
        if k >= 2:
            try:
                labels_sil = gmm.predict(X_sil)
                if len(np.unique(labels_sil)) >= 2:
                    sil = float(silhouette_score(X_sil, labels_sil))
            except Exception as exc:  # pragma: no cover
                logger.warning("Silhouette computation failed for K=%s: %s", k, exc)

        records.append(
            {
                "k": k,
                "aic": float(aic),
                "bic": float(bic),
                "val_mean_log_likelihood": val_ll,
                "silhouette_subsample": sil,
                "converged": bool(gmm.converged_),
                "lower_bound": float(gmm.lower_bound_),
                "n_iter": int(gmm.n_iter_),
            }
        )
        models[k] = gmm

    scores = pd.DataFrame(records).sort_values("k").reset_index(drop=True)
    scores.to_csv(out_dir / "cluster_bic_aic_scores.csv", index=False)

    best_k_bic = int(scores.loc[scores["bic"].idxmin(), "k"])
    best_k_aic = int(scores.loc[scores["aic"].idxmin(), "k"])
    best_k_val = int(scores.loc[scores["val_mean_log_likelihood"].idxmax(), "k"])
    best_k_sil = int(scores.loc[scores["silhouette_subsample"].idxmax(), "k"]) if scores["silhouette_subsample"].notna().any() else None

    logger.info("Best K by BIC: %s", best_k_bic)
    logger.info("Best K by AIC: %s", best_k_aic)
    logger.info("Best K by held-out validation log-likelihood: %s", best_k_val)
    logger.info("Best K by silhouette (subsample): %s", best_k_sil)

    if best_k_bic != best_k_val:
        logger.warning(
            "BIC-selected K (%s) disagrees with held-out validation K (%s). "
            "BIC can over-split with large N; inspect cluster_bic_aic.png and "
            "cluster_bic_aic_scores.csv before trusting the automatic pick.",
            best_k_bic,
            best_k_val,
        )

    # BIC remains the primary automatic criterion (it's still applied to a
    # sample, not the full population, so its complexity penalty is meaningful),
    # but the disagreement above is surfaced loudly rather than silently ignored.
    best_k = best_k_bic
    best_model = models[best_k]

    joblib.dump(best_model, out_dir / "cluster_gmm_best.joblib")
    joblib.dump(scaler, out_dir / "cluster_gmm_scaler.joblib")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(scores["k"], scores["bic"], marker="o", label="BIC")
    axes[0].plot(scores["k"], scores["aic"], marker="o", label="AIC")
    axes[0].axvline(best_k, color="k", linestyle="--", alpha=0.5, label=f"selected K={best_k}")
    axes[0].set_xlabel("Number of components (K)")
    axes[0].set_ylabel("Score (lower is better)")
    axes[0].set_title("BIC / AIC")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    ax2 = axes[1]
    ax2.plot(scores["k"], scores["val_mean_log_likelihood"], marker="o", color="tab:green", label="Held-out log-likelihood")
    ax2.set_xlabel("Number of components (K)")
    ax2.set_ylabel("Mean log-likelihood (higher is better)", color="tab:green")
    ax2.tick_params(axis="y", labelcolor="tab:green")
    ax2b = ax2.twinx()
    ax2b.plot(scores["k"], scores["silhouette_subsample"], marker="s", color="tab:purple", label="Silhouette")
    ax2b.set_ylabel("Silhouette (subsample)", color="tab:purple")
    ax2b.tick_params(axis="y", labelcolor="tab:purple")
    ax2.set_title("Held-out validation / silhouette")
    ax2.grid(True, alpha=0.3)

    fig.suptitle("Model selection diagnostics")
    fig.tight_layout()
    fig.savefig(out_dir / "cluster_bic_aic.png", dpi=200)
    plt.close(fig)

    return best_model, scaler, scores, best_k


def scan_current_max_label(cluster_ds: rasterio.io.DatasetReader) -> int:
    """Block-wise scan for the current max valid label (avoids loading the full raster)."""
    cluster_nodata = get_nodata_value(cluster_ds)
    running_max = None
    for _, window in cluster_ds.block_windows(1):
        block = cluster_ds.read(1, window=window)
        valid = np.ones(block.shape, dtype=bool)
        if not np.isnan(cluster_nodata):
            valid &= block != cluster_nodata
        if not np.any(valid):
            continue
        block_max = int(block[valid].max())
        running_max = block_max if running_max is None else max(running_max, block_max)
    if running_max is None:
        raise ValueError("Cluster raster contains no valid pixels.")
    return running_max


def determine_new_label_map(cluster_ds: rasterio.io.DatasetReader, n_subclusters: int) -> Tuple[np.ndarray, int]:
    """Assign fresh labels above the current max label so original IDs stay intact."""
    current_max = scan_current_max_label(cluster_ds)
    start_label = current_max + 1
    new_labels = np.arange(start_label, start_label + n_subclusters, dtype=np.int32)
    return new_labels, start_label


def predict_in_batches(model: GaussianMixture, feats: np.ndarray, batch_size: int) -> np.ndarray:
    """Predict in chunks to bound peak memory on unusually large raster blocks."""
    if feats.shape[0] <= batch_size:
        return model.predict(feats)
    out = np.empty(feats.shape[0], dtype=np.int64)
    for start in range(0, feats.shape[0], batch_size):
        end = start + batch_size
        out[start:end] = model.predict(feats[start:end])
    return out


def write_outputs(
    pca_ds: rasterio.io.DatasetReader,
    cluster_ds: rasterio.io.DatasetReader,
    target_cluster: int,
    model: GaussianMixture,
    scaler: StandardScaler,
    new_labels: np.ndarray,
    band_idx: np.ndarray,
    out_dir: Path,
    nodata_out: int,
    pred_batch_size: int,
) -> pd.DataFrame:
    profile = cluster_ds.profile.copy()
    profile.update(
        dtype="int32",
        count=1,
        nodata=nodata_out,
        compress="lzw",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

    merged_path = out_dir / "cluster_merged_map.tif"
    only_path = out_dir / "cluster_only_subclusters.tif"

    pca_nodata = get_nodata_value(pca_ds)
    cluster_nodata = get_nodata_value(cluster_ds)

    total_counts = np.zeros(new_labels.shape[0], dtype=np.int64)
    per_label_sum = np.zeros((new_labels.shape[0], band_idx.shape[0]), dtype=np.float64)

    with rasterio.open(merged_path, "w", **profile) as merged_ds, rasterio.open(only_path, "w", **profile) as only_ds:
        for _, window in cluster_ds.block_windows(1):
            pca, cluster = read_window_arrays(pca_ds, cluster_ds, window, band_idx)
            mask = make_valid_mask(pca, cluster, target_cluster, pca_nodata, cluster_nodata)

            merged = cluster.astype(np.int32, copy=True)
            only = np.full(cluster.shape, nodata_out, dtype=np.int32)

            if np.any(mask):
                feats = pca[mask]
                feats_scaled = scaler.transform(feats)
                sub_idx = predict_in_batches(model, feats_scaled, pred_batch_size)
                sub_labels = new_labels[sub_idx].astype(np.int32, copy=False)
                merged[mask] = sub_labels
                only[mask] = sub_labels

                for i in range(new_labels.shape[0]):
                    row_mask = sub_idx == i
                    c = int(row_mask.sum())
                    if c:
                        total_counts[i] += c
                        per_label_sum[i] += feats[row_mask].sum(axis=0)

            merged_ds.write(merged, 1, window=window)
            only_ds.write(only, 1, window=window)

    total_target = int(total_counts.sum())
    if total_target == 0:
        raise ValueError("Model prediction produced zero target pixels. Check your inputs.")

    means = []
    for i in range(new_labels.shape[0]):
        if total_counts[i] > 0:
            means.append(per_label_sum[i] / total_counts[i])
        else:
            means.append(np.full(band_idx.shape[0], np.nan, dtype=np.float64))

    rows = []
    for i, label in enumerate(new_labels):
        pct = 100.0 * total_counts[i] / total_target
        row = {
            "new_label": int(label),
            "pixel_count": int(total_counts[i]),
            "pct_within_target_cluster": float(pct),
            "small_subcluster_flag": bool(pct < 100.0 * MIN_SUBCLUSTER_FRACTION_DEFAULT),
        }
        for j, b in enumerate(band_idx):
            row[f"mean_pca_band_{int(b) + 1}"] = float(means[i][j]) if np.isfinite(means[i][j]) else np.nan
        rows.append(row)

    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "subcluster_summary.csv", index=False)

    small = summary[summary["small_subcluster_flag"]]
    if not small.empty:
        logger.warning(
            "%d sub-cluster(s) hold less than %.1f%% of the target-cluster pixels each: labels %s. "
            "These may be noise rather than real structure — inspect before using them downstream.",
            len(small),
            100.0 * MIN_SUBCLUSTER_FRACTION_DEFAULT,
            list(small["new_label"]),
        )

    return summary


def write_report(
    out_dir: Path,
    target_cluster: int,
    sample_shape: Tuple[int, int],
    scores: pd.DataFrame,
    summary: pd.DataFrame,
    best_k: int,
    new_labels: np.ndarray,
    start_label: int,
    band_idx: np.ndarray,
) -> None:
    best_bic_k = int(scores.loc[scores["bic"].idxmin(), "k"])
    best_aic_k = int(scores.loc[scores["aic"].idxmin(), "k"])
    best_val_k = int(scores.loc[scores["val_mean_log_likelihood"].idxmax(), "k"])

    report = []
    report.append(f"# Cluster {target_cluster} Hierarchical GMM Report\n")
    report.append(f"- PCA bands used (1-indexed): {[int(b) + 1 for b in band_idx]}")
    report.append(f"- Best K by BIC: {best_bic_k}")
    report.append(f"- Best K by AIC: {best_aic_k}")
    report.append(f"- Best K by held-out validation log-likelihood: {best_val_k}")
    report.append(f"- Selected K (BIC): {best_k}")
    if best_bic_k != best_val_k:
        report.append(
            "- **Warning:** BIC and held-out validation disagree on K. "
            "BIC's penalty weakens as sample size grows and can over-split. "
            "Review `cluster_bic_aic.png` before trusting the automatic pick."
        )
    report.append(f"- New label range: {int(start_label)}..{int(start_label + len(new_labels) - 1)}")
    report.append(f"- Training sample shape: {sample_shape[0]} x {sample_shape[1]} (standardized before fitting)")
    report.append(f"- Output merged raster: `{out_dir / 'cluster_merged_map.tif'}`")
    report.append(f"- Output target-only raster: `{out_dir / 'cluster_only_subclusters.tif'}`")
    report.append(f"- GMM model: `{out_dir / 'cluster_gmm_best.joblib'}`")
    report.append(f"- Feature scaler: `{out_dir / 'cluster_gmm_scaler.joblib'}` (required to reproduce predictions)")
    report.append(f"- Scores table: `{out_dir / 'cluster_bic_aic_scores.csv'}`")
    report.append(f"- Summary table: `{out_dir / 'subcluster_summary.csv'}`")
    report.append("")
    report.append("## Subcluster counts")
    report.append(
        summary[["new_label", "pixel_count", "pct_within_target_cluster", "small_subcluster_flag"]].to_markdown(index=False)
    )

    (out_dir / "cluster_report.md").write_text("\n".join(report), encoding="utf-8")

    meta = {
        "target_cluster": target_cluster,
        "selected_k": best_k,
        "best_k_bic": best_bic_k,
        "best_k_aic": best_aic_k,
        "best_k_validation": best_val_k,
        "new_label_start": int(start_label),
        "new_label_end": int(start_label + len(new_labels) - 1),
        "sample_rows": int(sample_shape[0]),
        "sample_cols": int(sample_shape[1]),
        "pca_bands_used_1indexed": [int(b) + 1 for b in band_idx],
        "subclusters": summary.to_dict(orient="records"),
    }
    (out_dir / "cluster_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")


# ============================================================
# CORE PIPELINE (callable directly from a notebook cell)
# ============================================================
def run(
    pca_path: str,
    cluster_path: str,
    target_cluster: int = TARGET_CLUSTER_DEFAULT,
    out_dir: Optional[str] = None,
    pca_bands: Optional[str] = None,
    k_min: int = K_MIN_DEFAULT,
    k_max: int = K_MAX_DEFAULT,
    sample_size: int = SEARCH_SAMPLE_SIZE_DEFAULT,
    validation_fraction: float = VALIDATION_FRACTION_DEFAULT,
    random_state: int = RANDOM_STATE_DEFAULT,
    covariance_type: str = COVARIANCE_TYPE_DEFAULT,
    reg_covar: float = REG_COVAR_DEFAULT,
    n_init: int = N_INIT_DEFAULT,
    max_iter: int = MAX_ITER_DEFAULT,
    pred_batch_size: int = PRED_BATCH_SIZE_DEFAULT,
    nodata_out: int = NODATA_OUT_DEFAULT,
) -> Path:
    """
    Run the full hierarchical-GMM recluster pipeline.

    On Kaggle, call this directly from a code cell — do NOT rely on
    argparse/main() there, since the notebook kernel injects its own
    arguments (e.g. '-f kernel-xxx.json') into sys.argv, which argparse
    will choke on. Example (Kaggle):

        pca_path = "/kaggle/input/<your-dataset>/11_PCA/PCA_Components.tif"
        cluster_path = "/kaggle/input/<your-dataset>/16_GMM/GMM_Cluster_Map.tif"
        out_dir = run(pca_path, cluster_path, target_cluster=12, k_max=8)

    Returns the output directory (under /kaggle/working on Kaggle) so you
    can zip/download it or chain further steps.
    """
    pca_path = Path(pca_path)
    cluster_path = Path(cluster_path)

    if not pca_path.exists():
        raise FileNotFoundError(
            f"PCA raster not found at {pca_path}. On Kaggle, browse the "
            f"'Input' panel on the right and copy the exact path under /kaggle/input/."
        )
    if not cluster_path.exists():
        raise FileNotFoundError(
            f"Cluster raster not found at {cluster_path}. On Kaggle, browse the "
            f"'Input' panel on the right and copy the exact path under /kaggle/input/."
        )

    if out_dir is None:
        out_dir_p = DEFAULT_OUT_ROOT / "gmm_recluster_3" / f"cluster_{target_cluster}_recluster"
    else:
        out_dir_p = Path(out_dir)
    out_dir_p.mkdir(parents=True, exist_ok=True)

    logger.info("PCA raster: %s", pca_path)
    logger.info("Cluster raster: %s", cluster_path)
    logger.info("Output dir: %s", out_dir_p)

    t0 = time.time()
    with rasterio.open(pca_path) as pca_ds, rasterio.open(cluster_path) as cluster_ds:
        ensure_alignment(pca_ds, cluster_ds)
        band_idx = parse_band_subset(pca_bands, pca_ds.count)
        logger.info("Using PCA bands (1-indexed): %s", [int(b) + 1 for b in band_idx])

        # 1) Build sample from the target cluster only.
        X = build_sample(
            pca_ds=pca_ds,
            cluster_ds=cluster_ds,
            target_cluster=target_cluster,
            sample_size=sample_size,
            random_state=random_state,
            band_idx=band_idx,
        )

        # 2) Fit GMM candidates (standardized), choose best K, save scaler + model.
        best_model, scaler, scores, best_k = fit_best_gmm(
            X=X,
            k_min=k_min,
            k_max=k_max,
            covariance_type=covariance_type,
            reg_covar=reg_covar,
            n_init=n_init,
            max_iter=max_iter,
            random_state=random_state,
            validation_fraction=validation_fraction,
            silhouette_sample_size=SILHOUETTE_SAMPLE_SIZE_DEFAULT,
            out_dir=out_dir_p,
        )

        # 3) Create fresh labels above the existing max label (block-wise scan).
        new_labels, start_label = determine_new_label_map(
            cluster_ds=cluster_ds,
            n_subclusters=best_model.n_components,
        )

        # 4) Re-write the full raster: replace target cluster by fresh subcluster IDs.
        summary = write_outputs(
            pca_ds=pca_ds,
            cluster_ds=cluster_ds,
            target_cluster=target_cluster,
            model=best_model,
            scaler=scaler,
            new_labels=new_labels,
            band_idx=band_idx,
            out_dir=out_dir_p,
            nodata_out=nodata_out,
            pred_batch_size=pred_batch_size,
        )

        # 5) Report.
        write_report(
            out_dir=out_dir_p,
            target_cluster=target_cluster,
            sample_shape=X.shape,
            scores=scores,
            summary=summary,
            best_k=best_k,
            new_labels=new_labels,
            start_label=start_label,
            band_idx=band_idx,
        )

    elapsed = time.time() - t0
    logger.info("Done in %.2f seconds", elapsed)
    logger.info("Merged raster: %s", out_dir_p / "cluster_merged_map.tif")
    logger.info("Target-only raster: %s", out_dir_p / "cluster_only_subclusters.tif")
    logger.info("Report: %s", out_dir_p / "cluster_report.md")
    return out_dir_p


# ============================================================
# CLI ENTRY POINT
# ============================================================
def main() -> None:
    """
    CLI wrapper — safe to invoke as `!python hierarchical_gmm_recluster.py --args`
    in a Kaggle cell. Uses parse_known_args so that if this file is instead
    executed inline in a Jupyter kernel (which injects its own '-f kernel.json'
    argv), unrecognized arguments are ignored rather than crashing the kernel.
    Prefer calling run(...) directly from a notebook cell instead.
    """
    parser = argparse.ArgumentParser(description="Hierarchical GMM reclustering for one GMM cluster.")
    parser.add_argument("--pca-path", type=str, required=False,
                         default=str(ROOT / "11_PCA" / "PCA_Components.tif"),
                         help="Path to PCA components raster")
    parser.add_argument("--cluster-path", type=str, required=False,
                         default=str(ROOT / "16_GMM" / "GMM_Cluster_Map.tif"),
                         help="Path to GMM cluster raster")
    parser.add_argument("--out-dir", type=str, default=None, help="Output directory (default: <working-dir>/gmm_recluster/cluster_<n>_recluster)")
    parser.add_argument("--target-cluster", type=int, default=TARGET_CLUSTER_DEFAULT, help="Cluster ID to split")
    parser.add_argument("--pca-bands", type=str, default=None, help="Comma-separated 1-indexed PCA bands to use, e.g. '1,2,3,4,5'. Default: all bands.")
    parser.add_argument("--k-min", type=int, default=K_MIN_DEFAULT, help="Minimum K to test")
    parser.add_argument("--k-max", type=int, default=K_MAX_DEFAULT, help="Maximum K to test")
    parser.add_argument("--sample-size", type=int, default=SEARCH_SAMPLE_SIZE_DEFAULT, help="Max sample size for GMM fit")
    parser.add_argument("--validation-fraction", type=float, default=VALIDATION_FRACTION_DEFAULT, help="Held-out fraction for validation log-likelihood")
    parser.add_argument("--random-state", type=int, default=RANDOM_STATE_DEFAULT, help="Random seed")
    parser.add_argument("--covariance-type", type=str, default=COVARIANCE_TYPE_DEFAULT, choices=["full", "tied", "diag", "spherical"], help="GMM covariance type")
    parser.add_argument("--reg-covar", type=float, default=REG_COVAR_DEFAULT, help="Regularization added to covariance diagonals (applied after standardization)")
    parser.add_argument("--n-init", type=int, default=N_INIT_DEFAULT, help="Number of initializations")
    parser.add_argument("--max-iter", type=int, default=MAX_ITER_DEFAULT, help="Maximum EM iterations")
    parser.add_argument("--pred-batch-size", type=int, default=PRED_BATCH_SIZE_DEFAULT, help="Batch size for prediction on raster blocks")
    parser.add_argument("--nodata-out", type=int, default=NODATA_OUT_DEFAULT, help="Nodata value for outputs")

    args, unknown = parser.parse_known_args()
    if unknown:
        logger.info("Ignoring unrecognized args (likely Jupyter kernel args): %s", unknown)

    run(
        pca_path=args.pca_path,
        cluster_path=args.cluster_path,
        target_cluster=args.target_cluster,
        out_dir=args.out_dir,
        pca_bands=args.pca_bands,
        k_min=args.k_min,
        k_max=args.k_max,
        sample_size=args.sample_size,
        validation_fraction=args.validation_fraction,
        random_state=args.random_state,
        covariance_type=args.covariance_type,
        reg_covar=args.reg_covar,
        n_init=args.n_init,
        max_iter=args.max_iter,
        pred_batch_size=args.pred_batch_size,
        nodata_out=args.nodata_out,
    )


if __name__ == "__main__":
    main()

## Step 6: Visual Progression & Results

The following screenshots illustrate the effectiveness of this Hierarchical Expert-in-the-Loop pipeline.

### 1. Initial Clustering (Without Mask)
<img src="https://i.ibb.co/bj0YdwRy/Whats-App-Image-2026-07-24-at-10-25-50-PM.jpg" width="800">

**Analysis Note**: Without an agricultural mask, the GMM distributions are heavily skewed by non-agricultural targets such as urban settlements, barren land, and water bodies. The crop boundaries are noisy, and distinct crop phenologies are overwhelmed by the macro-level landcover variance.

---

### 2. Primary Clustering (With Mask, K=20)
<img src="https://i.ibb.co/Df14QTGH/image.png" width="800">

**Analysis Note**: By constraining the analysis strictly to the expert-verified mask, the GMM forces its covariance matrices to model *only* vegetated cropland variations. However, at a high K value like K=20 globally, some clusters still represent "mixed" crop groups due to spectral overlap. A specific mixed cluster from this stage is selected as the `target_cluster` for hierarchical refinement.

---

### 3. Hierarchical Reclustering Results
<img src="https://i.ibb.co/Mxy8HzkG/image.png" width="800">
<br><br>
<img src="https://i.ibb.co/S44Yzx1w/image.png" width="800">

**Analysis Note**: By freezing a specific mixed target cluster and recursively applying a fresh GMM exclusively on its pixels, we free the covariance matrices to model the subtle, intra-class phenological differences. Whether at K=6 or K=10, the hierarchical reclustering successfully unmixes the target group into distinct, fine-grained sub-species variants that align cleanly with individual field boundaries.
